# Exp 06: Semi-Supervised Pseudo-Labeling for Unlabeled ADHD-200 Cohorts

## Metadata & Scientific Context
| Property | Specification |
| :--- | :--- |
| **Scientific Objective** | Leverage unlabeled imaging scans through confident semi-supervised pseudo-labeling to expand training cohorts. |
| **Reproducibility Status** | `Verified Executed Outputs Retained` |
| **Input Data Required** | Labeled training connectomes and unlabeled test/holdout connectomes. |
| **Primary Verified Artifacts** | `results/exp06/pseudo_labels_summary.csv, procedure comparison statistics.` |
| **Execution Environment** | `Track A (Semi-Supervised ML).` |

> [!NOTE]
> **Audit & Provenance Notice:** This notebook contains executed outputs preserved directly from the original scientific investigation. Paths have been made portable via environment variables (`ADHD200_*`). All numerical metrics and figures reflect the audited research artifacts.



In [1]:
import os
from pathlib import Path
import numpy as np

print("=" * 60)
print("CURRENT DIRECTORY AND FILE SEARCH")
print("=" * 60)

# Check current working directory
current_dir = Path.cwd()
print(f"Current directory: {current_dir}")

# List all files in current directory
print("\nFiles in current directory:")
for f in current_dir.iterdir():
    if f.is_dir():
        size = sum(f.stat().st_size for f in f.rglob('*')) / (1024**3) if f.is_dir() else 0
        print(f"  📁 {f.name}/ ({(size):.2f} GB)")
    else:
        print(f"  📄 {f.name} ({f.stat().st_size / (1024**2):.2f} MB)")

# Check parent directory
print(f"\nParent directory: {current_dir.parent}")
for f in current_dir.parent.iterdir():
    if f.is_dir():
        print(f"  📁 {f.name}/")

CURRENT DIRECTORY AND FILE SEARCH
Current directory: [DATA_ROOT]/RawDataBIDS

Files in current directory:
  📁 Brown/ (1.31 GB)
  📁 KKI/ (4.81 GB)
  📁 NYU/ (45.43 GB)
  📁 NeuroIMAGE/ (8.32 GB)
  📁 OHSU/ (4.12 GB)
  📁 Peking_1/ (5.85 GB)
  📁 Peking_2/ (3.21 GB)
  📁 Peking_3/ (2.18 GB)
  📁 Pittsburgh/ (3.59 GB)
  📁 WashU/ (3.08 GB)
  📄 .DS_Store (0.01 MB)
  📄 Brown_TestRelease_phenotypic.csv (0.00 MB)
  📄 KKI_phenotypic.csv (0.01 MB)
  📄 NYU_phenotypic.csv (0.02 MB)
  📄 NeuroIMAGE_phenotypic.csv (0.00 MB)
  📄 OHSU_TestRelease_phenotypic.csv (0.00 MB)
  📄 OHSU_phenotypic.csv (0.01 MB)
  📄 Peking_1_TestRelease_phenotypic.csv (0.00 MB)
  📄 Peking_1_phenotypic.csv (0.01 MB)
  📄 Peking_2_participants.tsv (0.01 MB)
  📄 Peking_3_participants.tsv (0.01 MB)
  📄 Pittsburgh_phenotypic.csv (0.01 MB)
  📄 desktop.ini (0.00 MB)
  📁 .ipynb_checkpoints/ (0.00 GB)
  📄 part1.ipynb (0.65 MB)
  📁 nilearn_cache/ (0.37 GB)
  📄 aal116NodeIndex.1D (0.00 MB)
  📄 aal116NodeNames.txt (0.00 MB)
  📄 aal116Reference.bi

In [2]:
import os
from pathlib import Path

# Common BIDS folder names
bids_patterns = [
    "raw_data", "RawData", "rawdata", "BIDS", "bids",
    "00_raw_data", "RawDataBIDS", "adhd200", "ADHD200"
]

found_dirs = []
for pattern in bids_patterns:
    matches = list(Path.cwd().glob(f"**/{pattern}"))
    matches += list(Path("/mnt").glob(f"**/{pattern}"))
    if matches:
        found_dirs.extend(matches)
        print(f"✅ Found: {pattern}")
        for m in matches[:3]:
            print(f"   {m}")

if not found_dirs:
    print("❌ No BIDS directories found in current path or /mnt")
    print("\nLet me search more broadly...")
    
    # Check common mount points
    for root_dir in ["/", "/mnt", "/home", "/data", "/mnt/ADHD200"]:
        path = Path(root_dir)
        if path.exists():
            print(f"\nSearching in: {root_dir}")
            for item in path.iterdir():
                if item.is_dir() and any(k in item.name.lower() for k in ["raw", "bids", "data", "adhd"]):
                    print(f"  📁 {item.name}/")

✅ Found: ADHD200
   /mnt/ADHD200


In [4]:
import os
from pathlib import Path

print("=" * 60)
print("SEARCHING FOR BOLD NIFTI FILES")
print("=" * 60)

# Search for .nii.gz files
nifti_files = list(Path.cwd().glob("**/*.nii.gz"))
print(f"Found {len(nifti_files)} NIfTI files")

if nifti_files:
    # Check if they're BOLD files
    bold_files = [f for f in nifti_files if "bold" in f.name.lower()]
    print(f"Found {len(bold_files)} BOLD files")
    for f in bold_files[:10]:
        print(f"  {f}")
else:
    print("No NIfTI files found in current directory")
    print("Searching in /mnt/ADHD200...")
    adhd_path = Path("/mnt/ADHD200")
    if adhd_path.exists():
        nifti_files = list(adhd_path.glob("**/*.nii.gz"))
        print(f"Found {len(nifti_files)} NIfTI files in /mnt/ADHD200")
        for f in nifti_files[:10]:
            print(f"  {f}")

SEARCHING FOR BOLD NIFTI FILES
Found 2363 NIfTI files
Found 1402 BOLD files
  [DATA_ROOT]/RawDataBIDS/Brown/sub-0026001/ses-1/func/sub-0026001_ses-1_task-rest_run-1_bold.nii.gz
  [DATA_ROOT]/RawDataBIDS/Brown/sub-0026002/ses-1/func/sub-0026002_ses-1_task-rest_run-1_bold.nii.gz
  [DATA_ROOT]/RawDataBIDS/Brown/sub-0026004/ses-1/func/sub-0026004_ses-1_task-rest_run-1_bold.nii.gz
  [DATA_ROOT]/RawDataBIDS/Brown/sub-0026005/ses-1/func/sub-0026005_ses-1_task-rest_run-1_bold.nii.gz
  [DATA_ROOT]/RawDataBIDS/Brown/sub-0026009/ses-1/func/sub-0026009_ses-1_task-rest_run-1_bold.nii.gz
  [DATA_ROOT]/RawDataBIDS/Brown/sub-0026014/ses-1/func/sub-0026014_ses-1_task-rest_run-1_bold.nii.gz
  [DATA_ROOT]/RawDataBIDS/Brown/sub-0026015/ses-1/func/sub-0026015_ses-1_task-rest_run-1_bold.nii.gz
  [DATA_ROOT]/RawDataBIDS/Brown/sub-0026016/ses-1/func/sub-0026016_ses-1_task-rest_run-1_bold.nii.gz
  [DATA_ROOT]/RawDataBIDS/Brown/sub-0026017/ses-1/func/sub-0026017_ses-1_task-rest_run-1_bold.nii.gz
  [DATA_ROOT]/R

In [3]:
from nilearn import datasets

# Option A: Use Harvard-Oxford atlas (already downloaded)
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')

# Option B: Use a local atlas file if you have it
# atlas = datasets.fetch_atlas_aal(data_dir='/path/to/your/local/atlas')

[fetch_atlas_harvard_oxford] Dataset found in /home/nvidia/nilearn_data/fsl

In [4]:
# THIS IS THE ONLY CODE YOU NEED TO RUN NOW
# No Nilearn Required - Only NumPy + scikit-learn

import numpy as np
from pathlib import Path
from tqdm import tqdm
from sklearn.cluster import KMeans
from scipy.stats import entropy

ROOT = Path("data")
TS_DIR = ROOT / "02_timeseries"

# Load your previously extracted ROI time series
ts_files = list(TS_DIR.glob("*_roi_timeseries.npy"))
print(f"Found {len(ts_files)} time series files")

all_timeseries = []
all_subjects = []

for f in ts_files:
    all_timeseries.append(np.load(f))
    all_subjects.append(f.stem.replace('_roi_timeseries', ''))

print(f"Loaded {len(all_timeseries)} subjects")
print(f"Time series shape: {all_timeseries[0].shape if all_timeseries else 'None'}")

# DYNAMIC FC (No Nilearn Used Here)
def compute_dynamic_fc(timeseries, window_size=30, step_size=5):
    T, N = timeseries.shape
    n_windows = (T - window_size) // step_size + 1
    dyn_fc = []
    for i in range(n_windows):
        start = i * step_size
        end = start + window_size
        window_ts = timeseries[start:end, :]
        fc = np.corrcoef(window_ts.T)
        np.fill_diagonal(fc, 0)
        dyn_fc.append(fc)
    return np.array(dyn_fc)

def compute_dynamic_features(dyn_fc, n_states=3):
    n_windows, N, _ = dyn_fc.shape
    upper_idx = np.triu_indices(N, k=1)
    dyn_flat = np.array([fc[upper_idx] for fc in dyn_fc])
    
    # Statistics
    mean_fc = np.mean(dyn_flat, axis=0)
    std_fc = np.std(dyn_flat, axis=0)
    var_fc = np.var(dyn_flat, axis=0)
    range_fc = np.max(dyn_flat, axis=0) - np.min(dyn_flat, axis=0)
    cv_fc = std_fc / (np.abs(mean_fc) + 1e-8)
    
    # Window-level metrics
    window_means = np.mean(dyn_flat, axis=1)
    window_stds = np.std(dyn_flat, axis=1)
    window_variance = np.var(dyn_flat, axis=1)
    
    # State analysis
    kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10)
    states = kmeans.fit_predict(dyn_flat)
    state_counts = np.bincount(states, minlength=n_states)
    state_fractions = state_counts / n_windows
    transitions = np.sum(states[:-1] != states[1:])
    transition_rate = transitions / (n_windows - 1)
    state_entropy = entropy(state_fractions + 1e-8)
    
    # Dwell times
    dwell_times = []
    for s in range(n_states):
        dwell_times.append(np.mean(states == s))
    
    features = np.concatenate([
        mean_fc,                    # n_edges
        std_fc,                     # n_edges
        var_fc,                     # n_edges
        range_fc,                   # n_edges
        cv_fc,                      # n_edges
        [np.mean(window_means)],    # 1
        [np.std(window_means)],     # 1
        [np.mean(window_stds)],     # 1
        [np.std(window_stds)],      # 1
        [np.mean(window_variance)], # 1
        [np.std(window_variance)],  # 1
        [transition_rate],          # 1
        [state_entropy],            # 1
        state_fractions,            # n_states
        dwell_times,                # n_states
    ])
    
    return features

# Run on all subjects
X_dynFC = []
for ts in tqdm(all_timeseries, desc="Computing dynamic FC"):
    dyn_fc = compute_dynamic_fc(ts)
    features = compute_dynamic_features(dyn_fc)
    X_dynFC.append(features)

X_dynFC = np.array(X_dynFC)
print(f"✅ Dynamic FC features: {X_dynFC.shape}")

# Save
MODALITY_DIR = ROOT / "05_modalities" / "dynamic_fc"
MODALITY_DIR.mkdir(parents=True, exist_ok=True)

np.save(MODALITY_DIR / "X_dynFC.npy", X_dynFC)
np.save(MODALITY_DIR / "dynFC_subjects.npy", np.array(all_subjects))

print(f"✅ Saved to {MODALITY_DIR}")

Found 955 time series files
Loaded 955 subjects
Time series shape: (251, 9170)


Computing dynamic FC:   0%|          | 0/955 [00:00<?, ?it/s]/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
Computing dynamic FC:   0%|          | 0/955 [00:58<?, ?it/s]


In [1]:
import numpy as np
from scipy.stats import entropy
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# STEP 1: Extract Sliding Windows
def extract_sliding_windows(timeseries, window_size=30, step_size=5):
    """Extract flattened dynamic FC windows for one subject."""
    T, N = timeseries.shape
    n_windows = (T - window_size) // step_size + 1
    
    if n_windows < 2:
        return None
    
    dyn_flat_list = []
    upper_idx = np.triu_indices(N, k=1)
    
    for i in range(n_windows):
        start = i * step_size
        end = start + window_size
        window_ts = timeseries[start:end, :]
        fc = np.corrcoef(window_ts.T)
        np.fill_diagonal(fc, 0)
        dyn_flat_list.append(fc[upper_idx])
    
    return np.array(dyn_flat_list)  # (n_windows, n_edges)

# STEP 2: Group-Level Clustering
def compute_group_centroids(pooled_dyn_flat, n_states_range=[2, 3, 4, 5]):
    """
    Finds cross-subject brain states by clustering pooled window data.
    Ensures State K means the exact same thing for every subject.
    """
    print(f"  Clustering global pool of {pooled_dyn_flat.shape[0]} windows...")
    
    best_k = 2
    best_silhouette = -1
    best_kmeans = None
    
    # Subsample if dataset is massive
    if pooled_dyn_flat.shape[0] > 50000:
        idx = np.random.choice(pooled_dyn_flat.shape[0], 50000, replace=False)
        clustering_data = pooled_dyn_flat[idx]
    else:
        clustering_data = pooled_dyn_flat

    for k in n_states_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(clustering_data)
        
        if len(np.unique(labels)) > 1:
            sample_size = min(5000, clustering_data.shape[0])
            sample_idx = np.random.choice(clustering_data.shape[0], sample_size, replace=False)
            silhouette = silhouette_score(clustering_data[sample_idx], labels[sample_idx])
            
            if silhouette > best_silhouette:
                best_silhouette = silhouette
                best_k = k
                best_kmeans = kmeans

    # Fallback
    if best_kmeans is None:
        best_k = 3
        best_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(clustering_data)
        
    print(f"  🏆 Selected Optimal State Count: K = {best_k} (Silhouette: {best_silhouette:.3f})")
    return best_kmeans

# STEP 3: Extract Features Per Subject
def extract_subject_metrics(dyn_flat, group_kmeans, n_edges):
    """
    Maps an individual subject's dynamic windows onto universal group states.
    """
    n_windows = dyn_flat.shape[0]
    
    if n_windows < 2:
        return None
    
    # Predict subject state labels using global group centroids
    subject_labels = group_kmeans.predict(dyn_flat)
    n_states = group_kmeans.n_clusters
    
    # State metrics
    state_counts = np.bincount(subject_labels, minlength=n_states)
    state_fractions = state_counts / n_windows
    
    # Transition probability matrix
    trans_matrix = np.zeros((n_states, n_states))
    for i in range(n_windows - 1):
        trans_matrix[subject_labels[i], subject_labels[i+1]] += 1
    trans_matrix = trans_matrix / (trans_matrix.sum(axis=1, keepdims=True) + 1e-8)
    
    # State entropy
    state_entropy = entropy(state_fractions + 1e-8)
    
    # Dwell times
    dwell_times = []
    for s in range(n_states):
        runs = np.where(np.diff(np.concatenate(([0], subject_labels == s, [0]))))[0]
        runs = runs.reshape(-1, 2)
        dwell = np.mean(runs[:, 1] - runs[:, 0]) if len(runs) > 0 else 0
        dwell_times.append(dwell)
        
    # Baseline statistical moments
    mean_fc = np.mean(dyn_flat, axis=0)
    std_fc = np.std(dyn_flat, axis=0)
    var_fc = np.var(dyn_flat, axis=0)
    range_fc = np.max(dyn_flat, axis=0) - np.min(dyn_flat, axis=0)
    cv_fc = std_fc / (np.abs(mean_fc) + 1e-8)
    
    window_means = np.mean(dyn_flat, axis=1)
    window_stds = np.std(dyn_flat, axis=1)
    
    # Core Feature Vector Assembly
    features = np.concatenate([
        mean_fc,                    # n_edges
        std_fc,                     # n_edges
        var_fc,                     # n_edges
        range_fc,                   # n_edges
        cv_fc,                      # n_edges
        [np.mean(window_means)],    # 1
        [np.std(window_means)],     # 1
        [np.mean(window_stds)],     # 1
        [np.std(window_stds)],      # 1
        state_fractions,            # K (Group-aligned!)
        [state_entropy],            # 1
        dwell_times,                # K (Group-aligned!)
        trans_matrix.flatten()      # K * K (Group-aligned!)
    ])
    
    return features

# STEP 4: Main Orchestrator
def compute_dynamic_fc_full_v2(timeseries_list, y, window_size=30, step_size=5, n_states_range=[2, 3, 4, 5]):
    """
    Complete dynamic FC pipeline with cross-subject clustering.
    
    Args:
        timeseries_list: List of (T, N) arrays
        y: Labels (0/1)
        window_size: Sliding window length
        step_size: Step between windows
        n_states_range: Candidate state counts
    
    Returns:
        X_features: (n_subjects, n_features)
        metadata: Dict with results
    """
    print("=" * 60)
    print("DYNAMIC FC PIPELINE (Cross-Subject States)")
    print("=" * 60)
    
    # PASS 1: Extract and Pool Windows
    print(f"\n[1/3] Extracting sliding windows (window={window_size}, step={step_size})...")
    all_subjects_flat_windows = []
    valid_indices = []
    n_edges = None
    
    for idx, ts in enumerate(tqdm(timeseries_list, desc="Extracting windows")):
        dyn_flat = extract_sliding_windows(ts, window_size, step_size)
        if dyn_flat is not None:
            all_subjects_flat_windows.append(dyn_flat)
            valid_indices.append(idx)
            if n_edges is None:
                n_edges = dyn_flat.shape[1]
    
    print(f"  Processed {len(valid_indices)} valid subjects")
    
    if len(all_subjects_flat_windows) == 0:
        print("  ❌ No valid windows found!")
        return None
    
    # PASS 2: Cluster Group & Project
    pooled_windows = np.vstack(all_subjects_flat_windows)
    print(f"  Pooled windows: {pooled_windows.shape}")
    
    print("\n[2/3] Computing group-level states...")
    group_kmeans = compute_group_centroids(pooled_windows, n_states_range)
    print(f"  States: {group_kmeans.n_clusters}")
    
    # PASS 3: Extract Features
    print("\n[3/3] Extracting features per subject...")
    X_features = []
    for dyn_flat in tqdm(all_subjects_flat_windows, desc="Extracting features"):
        features = extract_subject_metrics(dyn_flat, group_kmeans, n_edges)
        if features is not None:
            X_features.append(features)
    
    X_dynFC = np.array(X_features)
    
    # Results
    print("\n" + "=" * 60)
    print("RESULTS")
    print("=" * 60)
    print(f"  Subjects: {X_dynFC.shape[0]}")
    print(f"  Features: {X_dynFC.shape[1]}")
    print(f"  States: {group_kmeans.n_clusters}")
    
    return {
        'X_dynFC': X_dynFC,
        'n_subjects': X_dynFC.shape[0],
        'n_features': X_dynFC.shape[1],
        'n_states': group_kmeans.n_clusters,
        'group_kmeans': group_kmeans,
        'valid_indices': valid_indices
    }

# EXECUTION
# Assuming you have timeseries_list and y
# results = compute_dynamic_fc_full_v2(timeseries_list, y, window_size=30, step_size=5)
# if results:
# print(f"\n- Dynamic FC features shape: {results['X_dynFC'].shape}")
# X_dynFC = results['X_dynFC']
# np.save("/mnt/ADHD200/05_modalities/dynamic_fc/X_dynFC.npy", X_dynFC)

In [3]:
import numpy as np
import nibabel as nib
from nilearn import image, input_data, datasets
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("data")
RAW_DATA = ROOT / "RawDataBIDS"

print("=" * 60)
print("EXTRACTING ROI TIMESERIES")
print("=" * 60)

# Load atlas
print("Loading atlas...")
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
atlas_img = atlas.maps
print(f"✅ Atlas loaded: {len(atlas.labels) - 1} ROIs")

# Create masker
masker = input_data.NiftiLabelsMasker(
    atlas_img,
    standardize=True,
    detrend=True,
    low_pass=0.1,
    high_pass=0.01,
    t_r=2.0,
    memory='nilearn_cache',
    verbose=0
)

# Find BOLD files
bold_files = list(RAW_DATA.glob("**/*_bold.nii.gz"))
print(f"Found {len(bold_files)} BOLD files")

# Process first 20 subjects (test)
n_subjects = min(20, len(bold_files))
print(f"Processing {n_subjects} subjects...")

timeseries_list = []
subject_ids = []

for bold_file in tqdm(bold_files[:n_subjects], desc="Extracting"):
    try:
        # Extract subject ID
        subj_parts = bold_file.parts
        subj_id = None
        for part in subj_parts:
            if part.startswith('sub-'):
                subj_id = part.replace('sub-', '')
                break
        
        if subj_id is None:
            continue
        
        # Extract timeseries
        img = image.load_img(str(bold_file))
        ts = masker.fit_transform(img)
        
        timeseries_list.append(ts)
        subject_ids.append(subj_id)
        
        print(f"  ✅ {subj_id}: {ts.shape}")
        
    except Exception as e:
        print(f"  ❌ Error: {bold_file.name} - {str(e)[:80]}")

print(f"\n✅ Extracted {len(timeseries_list)} subjects")
print(f"Timeseries shape: {timeseries_list[0].shape if timeseries_list else 'None'}")

# Save for later use
if timeseries_list:
    TS_DIR = ROOT / "02_timeseries"
    TS_DIR.mkdir(parents=True, exist_ok=True)
    
    for subj, ts in zip(subject_ids, timeseries_list):
        np.save(TS_DIR / f"{subj}_roi_timeseries.npy", ts)
    
    print(f"✅ Saved to {TS_DIR}")

EXTRACTING ROI TIMESERIES
Loading atlas...


[fetch_atlas_harvard_oxford] Dataset found in /home/nvidia/nilearn_data/fsl

✅ Atlas loaded: 48 ROIs
Found 1402 BOLD files
Processing 20 subjects...


Extracting:   5%|▌         | 1/20 [00:02<00:43,  2.27s/it]

  ✅ 0026001: (251, 39)


...

  ✅ 0026043: (251, 48)


Extracting:  95%|█████████▌| 19/20 [00:41<00:02,  2.10s/it]

  ✅ 0026044: (251, 48)


Extracting: 100%|██████████| 20/20 [00:43<00:00,  2.18s/it]

  ✅ 0026045: (251, 47)

✅ Extracted 20 subjects
Timeseries shape: (251, 39)
✅ Saved to [DATA_ROOT]/02_timeseries


In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("data")

# Create the folder if it doesn't exist
fc_dir = ROOT / "03_fc_matrices"
fc_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ Created folder: {fc_dir}")

# Find the cohort file
cohort_files = list(ROOT.glob("**/master_cohort.csv"))
if not cohort_files:
    cohort_files = list(ROOT.glob("**/*cohort*.csv"))
    cohort_files += list(ROOT.glob("**/*phenotypic*.csv"))

if cohort_files:
    cohort_path = cohort_files[0]
    print(f"✅ Found cohort file: {cohort_path}")
    cohort = pd.read_csv(cohort_path)
    print(f"Cohort shape: {cohort.shape}")
    print(f"Columns: {cohort.columns.tolist()}")
    
    # Look for diagnosis column
    possible_dx_cols = ['DX', 'dx', 'diagnosis', 'adhd', 'group', 'label', 'Class']
    dx_col = None
    for col in possible_dx_cols:
        if col in cohort.columns:
            dx_col = col
            break
    
    if dx_col:
        print(f"✅ Using diagnosis column: {dx_col}")
        # Create binary labels (0 = Control, 1 = ADHD)
        y = (cohort[dx_col] != 0).astype(int).values
        print(f"Labels shape: {y.shape}")
        print(f"ADHD: {np.sum(y==1)}, Control: {np.sum(y==0)}")
        
        # Save to the newly created folder
        np.save(fc_dir / "y_binary.npy", y)
        print(f"✅ Saved labels to: {fc_dir / 'y_binary.npy'}")
    else:
        print("❌ No diagnosis column found.")
else:
    print("❌ No cohort file found.")

✅ Created folder: [DATA_ROOT]/03_fc_matrices
✅ Found cohort file: [DATA_ROOT]/RawDataBIDS/Brown_TestRelease_phenotypic.csv
Cohort shape: (26, 23)
Columns: ['ScanDir ID', 'Site', 'Gender', 'Age', 'Handedness', 'DX', 'Secondary Dx ', 'ADHD Measure', 'ADHD Index', 'Inattentive', 'Hyper/Impulsive', 'IQ Measure', 'Verbal IQ', 'Performance IQ', 'Full2 IQ', 'Full4 IQ', 'Med Status', 'QC_Rest_1', 'QC_Rest_2', 'QC_Rest_3', 'QC_Rest_4', 'QC_Anatomical_1', 'QC_Anatomical_2']
✅ Using diagnosis column: DX
Labels shape: (26,)
ADHD: 26, Control: 0
✅ Saved labels to: [DATA_ROOT]/03_fc_matrices/y_binary.npy


In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("data")

print("=" * 60)
print("FINDING ALL PHENOTYPIC FILES")
print("=" * 60)

# Search for all phenotypic files
pheno_files = list(ROOT.glob("**/*phenotypic*.csv"))
pheno_files += list(ROOT.glob("**/*participants*.csv"))
pheno_files += list(ROOT.glob("**/master_cohort.csv"))

# Remove duplicates
pheno_files = list(set(pheno_files))

print(f"Found {len(pheno_files)} files:\n")

for f in pheno_files:
    try:
        df = pd.read_csv(f)
        print(f"📄 {f.name}")
        print(f"   Path: {f.relative_to(ROOT) if f.is_relative_to(ROOT) else f}")
        print(f"   Shape: {df.shape}")
        
        # Check for diagnosis column
        dx_col = None
        for col in ['DX', 'dx', 'diagnosis', 'adhd', 'group', 'label']:
            if col in df.columns:
                dx_col = col
                break
        
        if dx_col:
            counts = df[dx_col].value_counts().to_dict()
            print(f"   DX column: {dx_col}")
            print(f"   Distribution: {counts}")
            
            # Count ADHD vs Control
            adhd_count = 0
            control_count = 0
            for key, val in counts.items():
                if key in [0, '0', 'Typically Developing Children', 'Control']:
                    control_count += val
                elif key in [1, 2, 3, '1', '2', '3', 'ADHD-Combined', 'ADHD-Inattentive', 'ADHD-Hyperactive/Impulsive']:
                    adhd_count += val
            print(f"   ✅ Control: {control_count}, ADHD: {adhd_count}")
        else:
            print(f"   ❌ No diagnosis column found")
        print()
        
    except Exception as e:
        print(f"   ❌ Error reading {f.name}: {e}\n")

FINDING ALL PHENOTYPIC FILES
Found 9 files:

📄 KKI_phenotypic.csv
   Path: RawDataBIDS/KKI_phenotypic.csv
   Shape: (83, 23)
   DX column: DX
   Distribution: {0: 61, 1: 16, 3: 5, 2: 1}
   ✅ Control: 61, ADHD: 22

📄 Peking_1_phenotypic.csv
   Path: RawDataBIDS/Peking_1_phenotypic.csv
   Shape: (85, 23)
   DX column: DX
   Distribution: {0: 61, 3: 17, 1: 7}
   ✅ Control: 61, ADHD: 24

📄 NYU_phenotypic.csv
   Path: RawDataBIDS/NYU_phenotypic.csv
   Shape: (222, 23)
   DX column: DX
   Distribution: {0: 99, 1: 77, 3: 44, 2: 2}
   ✅ Control: 99, ADHD: 123

📄 Peking_1_TestRelease_phenotypic.csv
   Path: RawDataBIDS/Peking_1_TestRelease_phenotypic.csv
   Shape: (51, 23)
   DX column: DX
   Distribution: {0: 27, 3: 14, 1: 9, 2: 1}
   ✅ Control: 27, ADHD: 24

📄 OHSU_phenotypic.csv
   Path: RawDataBIDS/OHSU_phenotypic.csv
   Shape: (79, 23)
   DX column: DX
   Distribution: {0: 42, 1: 23, 3: 12, 2: 2}
   ✅ Control: 42, ADHD: 37

📄 Brown_TestRelease_phenotypic.csv
   Path: RawDataBIDS/Brown_Test

In [14]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("data")

print("=" * 60)
print("COMBINING ALL PHENOTYPIC FILES")
print("=" * 60)

# Find ALL phenotypic files
pheno_files = list(ROOT.glob("**/*phenotypic*.csv"))
pheno_files += list(ROOT.glob("**/*participants*.csv"))

# Remove duplicates
pheno_files = list(set(pheno_files))

print(f"Found {len(pheno_files)} files")

# Combine all labels
all_labels = []
all_subjects = []
all_sites = []

for f in pheno_files:
    try:
        df = pd.read_csv(f)
        
        # Find ID column
        id_col = None
        for col in ['ScanDir ID', 'participant_id', 'subject_id', 'ID']:
            if col in df.columns:
                id_col = col
                break
        
        # Find DX column
        dx_col = None
        for col in ['DX', 'dx', 'diagnosis', 'adhd']:
            if col in df.columns:
                dx_col = col
                break
        
        # Find Site column
        site_col = None
        for col in ['Site', 'site']:
            if col in df.columns:
                site_col = col
                break
        
        if id_col is None or dx_col is None:
            print(f"⚠️ Skipping {f.name}: missing ID or DX column")
            continue
        
        # Extract data
        for idx, row in df.iterrows():
            subj_id = str(row[id_col])
            dx = row[dx_col]
            
            # Map DX to binary
            if dx in [0, '0', 'Typically Developing Children', 'Control']:
                label = 0
            elif dx in [1, 2, 3, '1', '2', '3', 
                        'ADHD-Combined', 'ADHD-Inattentive', 'ADHD-Hyperactive/Impulsive']:
                label = 1
            else:
                continue  # Skip unknown
            
            all_labels.append(label)
            all_subjects.append(subj_id)
            
            if site_col and site_col in df.columns:
                all_sites.append(str(row[site_col]))
            else:
                all_sites.append(f.name.replace('_phenotypic.csv', '').replace('_participants.csv', ''))
        
        print(f"✅ {f.name}: {len(df)} subjects")
        
    except Exception as e:
        print(f"❌ Error reading {f.name}: {e}")

# Convert to numpy arrays
y = np.array(all_labels)
subjects = np.array(all_subjects)
sites = np.array(all_sites)

print("\n" + "=" * 60)
print("FINAL COMBINED DATASET")
print("=" * 60)
print(f"Total subjects: {len(y)}")
print(f"ADHD: {np.sum(y==1)}")
print(f"Control: {np.sum(y==0)}")
print(f"Sites: {np.unique(sites)}")

# Save to 03_fc_matrices
fc_dir = ROOT / "03_fc_matrices"
fc_dir.mkdir(parents=True, exist_ok=True)

np.save(fc_dir / "y_binary.npy", y)
np.save(fc_dir / "subjects.npy", subjects)
np.save(fc_dir / "sites.npy", sites)

print(f"\n✅ Saved to {fc_dir}:")
print(f"  y_binary.npy: {y.shape}")
print(f"  subjects.npy: {subjects.shape}")
print(f"  sites.npy: {sites.shape}")

COMBINING ALL PHENOTYPIC FILES
Found 9 files
✅ KKI_phenotypic.csv: 83 subjects
✅ Peking_1_phenotypic.csv: 85 subjects
✅ NYU_phenotypic.csv: 222 subjects
✅ Peking_1_TestRelease_phenotypic.csv: 51 subjects
✅ OHSU_phenotypic.csv: 79 subjects
✅ Brown_TestRelease_phenotypic.csv: 26 subjects
✅ Pittsburgh_phenotypic.csv: 89 subjects
✅ OHSU_TestRelease_phenotypic.csv: 34 subjects
✅ NeuroIMAGE_phenotypic.csv: 48 subjects

FINAL COMBINED DATASET
Total subjects: 691
ADHD: 261
Control: 430
Sites: ['1' '3' '4.0' '5' '6' '6.0' '7']

✅ Saved to [DATA_ROOT]/03_fc_matrices:
  y_binary.npy: (691,)
  subjects.npy: (691,)
  sites.npy: (691,)


In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("data")

print("=" * 60)
print("LOADING COMPLETE DATASET")
print("=" * 60)

# 1. Load FC matrices (if available)
fc_path = ROOT / "03_fc_matrices" / "X_fc_norm.npy"
if fc_path.exists():
    X_fc = np.load(fc_path)
    print(f"✅ X_fc: {X_fc.shape}")
else:
    print("❌ X_fc.npy not found")
    X_fc = None

# 2. Load labels
y_path = ROOT / "03_fc_matrices" / "y_binary.npy"
if y_path.exists():
    y = np.load(y_path)
    print(f"✅ y: {y.shape}")
    print(f"   ADHD: {np.sum(y==1)}, Control: {np.sum(y==0)}")
else:
    print("❌ y_binary.npy not found")

# 3. Load subjects
subjects_path = ROOT / "03_fc_matrices" / "subjects.npy"
if subjects_path.exists():
    subjects = np.load(subjects_path, allow_pickle=True)
    print(f"✅ subjects: {len(subjects)}")
else:
    print("❌ subjects.npy not found")

# 4. Load sites
sites_path = ROOT / "03_fc_matrices" / "sites.npy"
if sites_path.exists():
    sites = np.load(sites_path, allow_pickle=True)
    print(f"✅ sites: {np.unique(sites)}")
else:
    print("❌ sites.npy not found")

LOADING COMPLETE DATASET
❌ X_fc.npy not found
✅ y: (691,)
   ADHD: 261, Control: 430
✅ subjects: 691
✅ sites: ['1' '3' '4.0' '5' '6' '6.0' '7']


In [17]:
from pathlib import Path
import numpy as np

ROOT = Path("data")

# Check RawDataBIDS
raw_bids = ROOT / "RawDataBIDS"
if raw_bids.exists():
    # Count subjects
    sub_folders = []
    for site in raw_bids.iterdir():
        if site.is_dir():
            subs = [f for f in site.iterdir() if f.is_dir() and f.name.startswith('sub-')]
            print(f"{site.name}: {len(subs)} subjects")
            sub_folders.extend(subs)
    print(f"\nTotal subjects in RawDataBIDS: {len(sub_folders)}")
else:
    print("RawDataBIDS not found")

Brown: 26 subjects
KKI: 83 subjects
NYU: 263 subjects
NeuroIMAGE: 73 subjects
OHSU: 113 subjects
Peking_1: 136 subjects
Peking_2: 67 subjects
Peking_3: 42 subjects
Pittsburgh: 98 subjects
WashU: 60 subjects
.ipynb_checkpoints: 0 subjects
nilearn_cache: 0 subjects

Total subjects in RawDataBIDS: 961


In [18]:
import numpy as np
import nibabel as nib
from nilearn import image, input_data, datasets
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("data")
RAW_DATA = ROOT / "RawDataBIDS"

print("=" * 60)
print("EXTRACTING ROI TIMESERIES FOR ALL SUBJECTS")
print("=" * 60)

# Load atlas
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
atlas_img = atlas.maps
print(f"✅ Atlas loaded: {len(atlas.labels) - 1} ROIs")

# Create masker
masker = input_data.NiftiLabelsMasker(
    atlas_img,
    standardize=True,
    detrend=True,
    low_pass=0.1,
    high_pass=0.01,
    t_r=2.0,
    memory='nilearn_cache',
    verbose=0
)

# Find ALL BOLD files
bold_files = list(RAW_DATA.glob("**/*_bold.nii.gz"))
print(f"Found {len(bold_files)} BOLD files")

# Process ALL subjects
all_timeseries = []
all_subject_ids = []
failed = []

for bold_file in tqdm(bold_files, desc="Extracting"):
    try:
        # Extract subject ID
        subj_id = None
        for part in bold_file.parts:
            if part.startswith('sub-'):
                subj_id = part.replace('sub-', '')
                break
        
        if subj_id is None:
            continue
        
        # Extract timeseries
        img = image.load_img(str(bold_file))
        ts = masker.fit_transform(img)
        
        all_timeseries.append(ts)
        all_subject_ids.append(subj_id)
        
    except Exception as e:
        failed.append(bold_file.name)

print(f"\n✅ Extracted {len(all_timeseries)} subjects")
print(f"❌ Failed: {len(failed)}")

if all_timeseries:
    print(f"Timeseries shape: {all_timeseries[0].shape}")
    
    # Save
    TS_DIR = ROOT / "02_timeseries"
    TS_DIR.mkdir(parents=True, exist_ok=True)
    
    for subj, ts in zip(all_subject_ids, all_timeseries):
        np.save(TS_DIR / f"{subj}_roi_timeseries.npy", ts)
    
    print(f"✅ Saved to {TS_DIR}")

EXTRACTING ROI TIMESERIES FOR ALL SUBJECTS


[fetch_atlas_harvard_oxford] Dataset found in /home/nvidia/nilearn_data/fsl

✅ Atlas loaded: 48 ROIs
Found 1402 BOLD files


Extracting:   1%|          | 11/1402 [00:14<29:32,  1.27s/it]


In [21]:
from pathlib import Path
import nibabel as nib

ROOT = Path("data")

# Look for AAL files
aal_files = list(ROOT.glob("**/*aal*.nii*"))
aal_files += list(ROOT.glob("**/*AAL*.nii*"))
aal_files += list(ROOT.glob("**/*atlas*.nii*"))

print("=" * 60)
print("FINDING AAL ATLAS FILES")
print("=" * 60)

for f in aal_files:
    print(f"  {f.relative_to(ROOT) if f.is_relative_to(ROOT) else f}")

# Check specific location
raw_bids = ROOT / "RawBIDS"
if raw_bids.exists():
    print(f"\n✅ RawBIDS folder exists")
    atlas_files = list(raw_bids.glob("*.nii*")) + list(raw_bids.glob("*.gz"))
    print(f"Files in RawBIDS:")
    for f in atlas_files:
        print(f"  {f.name}")
        # Try to load
        try:
            img = nib.load(f)
            print(f"    Shape: {img.shape}")
            print(f"    Data type: {img.get_data_dtype()}")
        except Exception as e:
            print(f"    Error loading: {e}")

FINDING AAL ATLAS FILES
  RawDataBIDS/aal116MNI.nii.gz


In [ ]:
import numpy as np
import nibabel as nib
from nilearn import image, input_data
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("data")

# STEP 1: LOAD AAL ATLAS (116 ROIs)
atlas_path = ROOT / "RawDataBIDS" / "aal116MNI.nii.gz"
print("=" * 60)
print("LOADING AAL ATLAS")
print("=" * 60)

if not atlas_path.exists():
    print(f"❌ Atlas not found at: {atlas_path}")
    exit()

atlas_img = nib.load(atlas_path)
data = atlas_img.get_fdata()
target_n_rois = int(np.max(data))
print(f"✅ Target ROIs: {target_n_rois}")

# STEP 2: CREATE MASKER
masker = input_data.NiftiLabelsMasker(
    atlas_img,
    standardize=True,
    detrend=True,
    low_pass=0.1,
    high_pass=0.01,
    t_r=2.0,
    memory='nilearn_cache',
    verbose=0
)
print("✅ Masker created")

# STEP 3: FIND ALL BOLD FILES
RAW_DATA = ROOT / "RawDataBIDS"
bold_files = list(RAW_DATA.glob("**/*_bold.nii.gz"))
print(f"\n📁 Found {len(bold_files)} BOLD files")

# STEP 4: EXTRACT ROI TIMESERIES
print("\n" + "=" * 60)
print("EXTRACTING ROI TIMESERIES")
print("=" * 60)

all_timeseries = []
all_subject_ids = []
failed = []

for bold_file in tqdm(bold_files, desc="Extracting"):
    try:
        # Extract subject ID
        subj_id = None
        for part in bold_file.parts:
            if part.startswith('sub-'):
                subj_id = part.replace('sub-', '')
                break
        
        if subj_id is None:
            continue
        
        # Load and extract
        img = image.load_img(str(bold_file))
        ts = masker.fit_transform(img)
        
        # Check shape - if not 116 ROIs, pad with zeros
        if ts.shape[1] < target_n_rois:
            padded_ts = np.zeros((ts.shape[0], target_n_rois))
            padded_ts[:, :ts.shape[1]] = ts
            ts = padded_ts
            print(f"  ⚠️ {subj_id}: {ts.shape[1]} → {target_n_rois} (padded)")
        elif ts.shape[1] > target_n_rois:
            ts = ts[:, :target_n_rois]  # Truncate if more ROIs
            print(f"  ⚠️ {subj_id}: {ts.shape[1]} → {target_n_rois} (truncated)")
        
        all_timeseries.append(ts)
        all_subject_ids.append(subj_id)
        
    except Exception as e:
        failed.append(bold_file.name)

print(f"\n✅ Extracted {len(all_timeseries)} subjects")
print(f"❌ Failed: {len(failed)}")

# STEP 5: COMPUTE FC MATRICES
print("\n" + "=" * 60)
print("COMPUTING FC MATRICES")
print("=" * 60)

X_fc = []
valid_subjects = []

for ts, subj_id in tqdm(zip(all_timeseries, all_subject_ids), total=len(all_timeseries), desc="Computing FC"):
    try:
        fc = np.corrcoef(ts.T)
        np.fill_diagonal(fc, 0)
        X_fc.append(fc)
        valid_subjects.append(subj_id)
    except Exception as e:
        print(f"  ❌ {subj_id}: {e}")

# Convert to array - now all should have same shape
X_fc = np.array(X_fc)
print(f"✅ FC matrices: {X_fc.shape}")

# STEP 6: SAVE
TS_DIR = ROOT / "02_timeseries"
TS_DIR.mkdir(parents=True, exist_ok=True)

print("\nSaving timeseries...")
for subj, ts in tqdm(zip(valid_subjects, all_timeseries), total=len(valid_subjects), desc="Saving"):
    np.save(TS_DIR / f"{subj}_roi_timeseries.npy", ts)

FC_DIR = ROOT / "03_fc_matrices"
FC_DIR.mkdir(parents=True, exist_ok=True)

np.save(FC_DIR / "X_fc.npy", X_fc)
np.save(FC_DIR / "X_fc_subjects.npy", np.array(valid_subjects))

print(f"✅ Saved to {FC_DIR}")
print(f"   X_fc: {X_fc.shape}")

LOADING AAL ATLAS
✅ Target ROIs: 9170
✅ Masker created

📁 Found 1402 BOLD files

EXTRACTING ROI TIMESERIES


Extracting:   0%|          | 1/1402 [00:00<22:10,  1.05it/s]

  ⚠️ 0026001: 9170 → 9170 (padded)


Extracting:   0%|          | 2/1402 [00:01<22:03,  1.06it/s]

  ⚠️ 0026002: 9170 → 9170 (padded)


...

Extracting: 100%|█████████▉| 1401/1402 [1:14:35<00:02,  2.22s/it]

  ⚠️ 0015062: 9170 → 9170 (padded)


Extracting: 100%|██████████| 1402/1402 [1:14:37<00:00,  3.19s/it]


  ⚠️ 0015062: 9170 → 9170 (padded)

✅ Extracted 1402 subjects
❌ Failed: 0

COMPUTING FC MATRICES


Computing FC: 100%|██████████| 1402/1402 [41:51<00:00,  1.79s/it] 


In [3]:
from pathlib import Path
import numpy as np

ROOT = Path("data")
TS_DIR = ROOT / "02_timeseries"

print("=" * 60)
print("CHECKING TIMESERIES FILES")
print("=" * 60)

if TS_DIR.exists():
    ts_files = list(TS_DIR.glob("*_roi_timeseries.npy"))
    print(f"Found {len(ts_files)} timeseries files")
    
    if ts_files:
        print(f"First 5 files:")
        for f in ts_files[:5]:
            print(f"  {f.name}")
        
        # Load one sample
        sample = np.load(ts_files[0])
        print(f"\nSample shape: {sample.shape}")
    else:
        print("❌ No .npy files found!")
else:
    print(f"❌ Directory not found: {TS_DIR}")

CHECKING TIMESERIES FILES
Found 20 timeseries files
First 5 files:
  0026001_roi_timeseries.npy
  0026002_roi_timeseries.npy
  0026004_roi_timeseries.npy
  0026005_roi_timeseries.npy
  0026009_roi_timeseries.npy

Sample shape: (251, 93)


In [4]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

ROOT = Path("data")
FC_DIR = ROOT / "03_fc_matrices"

# Load the full FC matrices (1402 subjects)
X_fc_full = np.load(FC_DIR / "X_fc.npy")
fc_subjects_full = np.load(FC_DIR / "X_fc_subjects.npy", allow_pickle=True)

print(f"Full FC: {X_fc_full.shape}")

# Create mapping from subject to index
subject_to_idx = {subj: i for i, subj in enumerate(fc_subjects_full)}

# Get unique subjects from timeseries
TS_DIR = ROOT / "02_timeseries"
ts_files = list(TS_DIR.glob("*_roi_timeseries.npy"))

unique_subjects = set()
for f in ts_files:
    subj_id = f.stem.replace('_roi_timeseries', '')
    unique_subjects.add(subj_id)

print(f"Unique subjects: {len(unique_subjects)}")

# Extract FC matrices for unique subjects
X_fc_unique = []
valid_subjects = []

for subj in tqdm(unique_subjects, desc="Extracting unique FC"):
    if subj in subject_to_idx:
        idx = subject_to_idx[subj]
        X_fc_unique.append(X_fc_full[idx])
        valid_subjects.append(subj)

X_fc_unique = np.array(X_fc_unique)
print(f"✅ Unique FC: {X_fc_unique.shape}")

In [5]:
from pathlib import Path
import numpy as np

ROOT = Path("data")
FC_DIR = ROOT / "03_fc_matrices"

print("=" * 60)
print("FILES IN 03_fc_matrices")
print("=" * 60)

if FC_DIR.exists():
    files = list(FC_DIR.glob("*"))
    print(f"Found {len(files)} files:")
    for f in files:
        size_mb = f.stat().st_size / (1024 * 1024) if f.is_file() else 0
        print(f"  {f.name} ({size_mb:.2f} MB)" if f.is_file() else f"  {f.name}/")
else:
    print(f"❌ {FC_DIR} does not exist!")
    FC_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✅ Created {FC_DIR}")

FILES IN 03_fc_matrices
Found 3 files:
  y_binary.npy (0.01 MB)
  subjects.npy (0.02 MB)
  sites.npy (0.01 MB)


In [8]:
from pathlib import Path

ROOT = Path("data")
RAW_DATA = ROOT / "RawDataBIDS"

bold_files = list(RAW_DATA.glob("**/*_bold.nii.gz"))
print(f"Total BOLD files: {len(bold_files)}")

# Group by subject
subjects = set()
for f in bold_files:
    for part in f.parts:
        if part.startswith('sub-'):
            subjects.add(part.replace('sub-', ''))
            break

print(f"Unique subjects: {len(subjects)}")

Total BOLD files: 1402
Unique subjects: 955


In [9]:
import numpy as np
import nibabel as nib
from nilearn import image, input_data
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("data")
RAW_DATA = ROOT / "RawDataBIDS"

# Load AAL atlas
atlas_path = ROOT / "RawDataBIDS" / "aal116MNI.nii.gz"
if not atlas_path.exists():
    print(f"❌ Atlas not found!")
    exit()

print("=" * 60)
print("LOADING AAL ATLAS")
print("=" * 60)
atlas_img = nib.load(atlas_path)
data = atlas_img.get_fdata()
target_n_rois = int(np.max(data))
print(f"✅ Target ROIs: {target_n_rois}")

# Create masker
masker = input_data.NiftiLabelsMasker(
    atlas_img,
    standardize=True,
    detrend=True,
    low_pass=0.1,
    high_pass=0.01,
    t_r=2.0,
    memory='nilearn_cache',
    verbose=0
)
print("✅ Masker created")

# Find all BOLD files
bold_files = list(RAW_DATA.glob("**/*_bold.nii.gz"))
print(f"\n📁 Found {len(bold_files)} BOLD files")

# Group by subject to avoid duplicates
subject_bold_map = {}
for f in bold_files:
    subj_id = None
    for part in f.parts:
        if part.startswith('sub-'):
            subj_id = part.replace('sub-', '')
            break
    if subj_id:
        if subj_id not in subject_bold_map:
            subject_bold_map[subj_id] = []
        subject_bold_map[subj_id].append(f)

print(f"✅ Found {len(subject_bold_map)} unique subjects")

# Select one file per subject (prefer ses-1/run-1)
unique_bold_files = []
for subj_id, files in subject_bold_map.items():
    selected = None
    for f in files:
        parts = str(f)
        if 'ses-1' in parts and 'run-1' in parts:
            selected = f
            break
    if selected is None:
        selected = files[0]
    unique_bold_files.append(selected)

print(f"✅ Selected {len(unique_bold_files)} unique BOLD files")

# EXTRACT ROI TIMESERIES FOR ALL SUBJECTS
print("\n" + "=" * 60)
print("EXTRACTING ROI TIMESERIES (ALL SUBJECTS)")
print("=" * 60)

TS_DIR = ROOT / "02_timeseries"
TS_DIR.mkdir(parents=True, exist_ok=True)

all_timeseries = []
all_subject_ids = []
failed = []

for bold_file in tqdm(unique_bold_files, desc="Extracting"):
    try:
        # Extract subject ID
        subj_id = None
        for part in bold_file.parts:
            if part.startswith('sub-'):
                subj_id = part.replace('sub-', '')
                break
        
        if subj_id is None:
            continue
        
        # Load and extract
        img = image.load_img(str(bold_file))
        ts = masker.fit_transform(img)
        
        # Pad/truncate to target ROIs
        if ts.shape[1] < target_n_rois:
            padded_ts = np.zeros((ts.shape[0], target_n_rois))
            padded_ts[:, :ts.shape[1]] = ts
            ts = padded_ts
        elif ts.shape[1] > target_n_rois:
            ts = ts[:, :target_n_rois]
        
        all_timeseries.append(ts)
        all_subject_ids.append(subj_id)
        
        # Save immediately to avoid losing progress
        np.save(TS_DIR / f"{subj_id}_roi_timeseries.npy", ts)
        
    except Exception as e:
        failed.append(bold_file.name)
        if len(failed) % 10 == 1:
            print(f"  ❌ {bold_file.name}: {str(e)[:60]}")

print(f"\n✅ Extracted {len(all_timeseries)} subjects")
print(f"❌ Failed: {len(failed)}")

# Save subject list
np.save(TS_DIR / "all_subjects.npy", np.array(all_subject_ids))

print(f"✅ Timeseries saved to {TS_DIR}")
print(f"   Total subjects: {len(all_subject_ids)}")

LOADING AAL ATLAS
✅ Target ROIs: 9170
✅ Masker created

📁 Found 1402 BOLD files
✅ Found 955 unique subjects
✅ Selected 955 unique BOLD files

EXTRACTING ROI TIMESERIES (ALL SUBJECTS)


Extracting: 100%|██████████| 955/955 [14:09<00:00,  1.12it/s]


✅ Extracted 955 subjects
❌ Failed: 0
✅ Timeseries saved to [DATA_ROOT]/02_timeseries
   Total subjects: 955


In [10]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("data")
TS_DIR = ROOT / "02_timeseries"
FC_DIR = ROOT / "03_fc_matrices"

print("=" * 60)
print("COMPUTING FC MATRICES (ALL 955 SUBJECTS)")
print("=" * 60)

# Find all timeseries files
ts_files = list(TS_DIR.glob("*_roi_timeseries.npy"))
print(f"Found {len(ts_files)} timeseries files")

if len(ts_files) == 0:
    print("❌ No timeseries files found!")
    exit()

TARGET_ROIS = 116  # AAL atlas has 116 ROIs
print(f"Target ROIs: {TARGET_ROIS}")

X_fc_list = []
fc_subjects = []
failed = []
inconsistent_shapes = []

for ts_file in tqdm(ts_files, desc="Computing FC"):
    try:
        ts = np.load(ts_file)
        
        # Check if timeseries has correct number of ROIs
        n_rois = ts.shape[1]
        if n_rois != TARGET_ROIS:
            inconsistent_shapes.append((ts_file.name, n_rois))
            # Pad or truncate to target shape
            if n_rois < TARGET_ROIS:
                padded_ts = np.zeros((ts.shape[0], TARGET_ROIS))
                padded_ts[:, :n_rois] = ts
                ts = padded_ts
            elif n_rois > TARGET_ROIS:
                ts = ts[:, :TARGET_ROIS]
        
        # Compute correlation matrix (FC)
        fc = np.corrcoef(ts.T)
        np.fill_diagonal(fc, 0)
        
        # Extract subject ID
        subj_id = ts_file.stem.replace('_roi_timeseries', '')
        
        X_fc_list.append(fc)
        fc_subjects.append(subj_id)
        
    except Exception as e:
        failed.append((ts_file.name, str(e)))

print(f"\n✅ Processed {len(X_fc_list)} subjects")
print(f"❌ Failed: {len(failed)}")

if inconsistent_shapes:
    print(f"⚠️ Inconsistent ROIs: {len(inconsistent_shapes)} subjects")
    print(f"   Examples: {inconsistent_shapes[:5]}")

# Convert to numpy array - now all should be (116, 116)
X_fc = np.array(X_fc_list)
fc_subjects = np.array(fc_subjects)

print(f"\n✅ FC matrices shape: {X_fc.shape}")
print(f"✅ Subjects: {len(fc_subjects)}")

# Save
FC_DIR.mkdir(parents=True, exist_ok=True)
np.save(FC_DIR / "X_fc.npy", X_fc)
np.save(FC_DIR / "X_fc_subjects.npy", fc_subjects)

print(f"\n✅ Saved to {FC_DIR}")
print(f"   X_fc.npy: {X_fc.shape}")
print(f"   X_fc_subjects.npy: {fc_subjects.shape}")

COMPUTING FC MATRICES (ALL 955 SUBJECTS)
Found 955 timeseries files
Target ROIs: 116


Computing FC: 100%|██████████| 955/955 [00:02<00:00, 450.43it/s]


✅ Processed 955 subjects
❌ Failed: 0
⚠️ Inconsistent ROIs: 955 subjects
   Examples: [('0026001_roi_timeseries.npy', 9170), ('0026002_roi_timeseries.npy', 9170), ('0026004_roi_timeseries.npy', 9170), ('0026005_roi_timeseries.npy', 9170), ('0026009_roi_timeseries.npy', 9170)]

✅ FC matrices shape: (955, 116, 116)
✅ Subjects: 955

✅ Saved to [DATA_ROOT]/03_fc_matrices
   X_fc.npy: (955, 116, 116)
   X_fc_subjects.npy: (955,)


In [11]:
import numpy as np
from pathlib import Path

ROOT = Path("data")
FC_DIR = ROOT / "03_fc_matrices"

print("=" * 60)
print("ALIGNING FC MATRICES WITH LABELS")
print("=" * 60)

# Load FC matrices
X_fc = np.load(FC_DIR / "X_fc.npy")
fc_subjects = np.load(FC_DIR / "X_fc_subjects.npy", allow_pickle=True)

print(f"FC subjects: {len(fc_subjects)}")
print(f"FC shape: {X_fc.shape}")

# Load labels
y_all = np.load(FC_DIR / "y_binary.npy")
subjects_all = np.load(FC_DIR / "subjects.npy", allow_pickle=True)
sites_all = np.load(FC_DIR / "sites.npy", allow_pickle=True)

print(f"Label subjects: {len(subjects_all)}")
print(f"  ADHD: {np.sum(y_all == 1)}")
print(f"  Control: {np.sum(y_all == 0)}")

# Create mapping
label_map = {subj: i for i, subj in enumerate(subjects_all)}
site_map = {subj: i for i, subj in enumerate(subjects_all)}

# Align
aligned_fc = []
aligned_labels = []
aligned_subjects = []
aligned_sites = []

for subj, fc in zip(fc_subjects, X_fc):
    if subj in label_map:
        idx = label_map[subj]
        aligned_fc.append(fc)
        aligned_labels.append(y_all[idx])
        aligned_subjects.append(subj)
        aligned_sites.append(sites_all[idx] if subj in site_map else "Unknown")

aligned_fc = np.array(aligned_fc)
aligned_labels = np.array(aligned_labels)
aligned_sites = np.array(aligned_sites)

print("\n" + "=" * 60)
print("FINAL ALIGNED DATASET")
print("=" * 60)
print(f"Subjects: {len(aligned_subjects)}")
print(f"FC shape: {aligned_fc.shape}")
print(f"ADHD: {np.sum(aligned_labels == 1)}")
print(f"Control: {np.sum(aligned_labels == 0)}")
print(f"Sites: {np.unique(aligned_sites)}")

# Site distribution
print("\nSite distribution:")
for site in np.unique(aligned_sites):
    mask = aligned_sites == site
    print(f"  {site}: {np.sum(mask)} (ADHD: {np.sum(aligned_labels[mask] == 1)}, Control: {np.sum(aligned_labels[mask] == 0)})")

# Save aligned data
np.save(FC_DIR / "X_fc_aligned.npy", aligned_fc)
np.save(FC_DIR / "y_aligned.npy", aligned_labels)
np.save(FC_DIR / "aligned_subjects.npy", np.array(aligned_subjects))
np.save(FC_DIR / "aligned_sites.npy", aligned_sites)

print(f"\n✅ Aligned data saved to {FC_DIR}")
print(f"   X_fc_aligned.npy: {aligned_fc.shape}")
print(f"   y_aligned.npy: {aligned_labels.shape}")
print(f"   aligned_subjects.npy: {aligned_subjects.shape}")
print(f"   aligned_sites.npy: {aligned_sites.shape}")

ALIGNING FC MATRICES WITH LABELS
FC subjects: 955
FC shape: (955, 116, 116)
Label subjects: 691
  ADHD: 261
  Control: 430

FINAL ALIGNED DATASET
Subjects: 391
FC shape: (391, 116, 116)
ADHD: 161
Control: 230
Sites: ['1' '3' '5' '6']

Site distribution:
  1: 136 (ADHD: 48, Control: 88)
  3: 83 (ADHD: 22, Control: 61)
  5: 93 (ADHD: 54, Control: 39)
  6: 79 (ADHD: 37, Control: 42)

✅ Aligned data saved to [DATA_ROOT]/03_fc_matrices
   X_fc_aligned.npy: (391, 116, 116)
   y_aligned.npy: (391,)


In [12]:
import numpy as np
from pathlib import Path

ROOT = Path("data")
FC_DIR = ROOT / "03_fc_matrices"

# Load labels and subjects
y_all = np.load(FC_DIR / "y_binary.npy")
subjects_all = np.load(FC_DIR / "subjects.npy", allow_pickle=True)
sites_all = np.load(FC_DIR / "sites.npy", allow_pickle=True)

print("=" * 60)
print("LABELLED SUBJECTS BY SITE")
print("=" * 60)

unique_sites = np.unique(sites_all)
for site in unique_sites:
    mask = sites_all == site
    count = np.sum(mask)
    adhd = np.sum(y_all[mask] == 1)
    control = np.sum(y_all[mask] == 0)
    print(f"  Site {site}: {count} subjects (ADHD: {adhd}, Control: {control})")

LABELLED SUBJECTS BY SITE
  Site 1: 136 subjects (ADHD: 48, Control: 88)
  Site 3: 83 subjects (ADHD: 22, Control: 61)
  Site 4.0: 48 subjects (ADHD: 25, Control: 23)
  Site 5: 222 subjects (ADHD: 123, Control: 99)
  Site 6: 79 subjects (ADHD: 37, Control: 42)
  Site 6.0: 34 subjects (ADHD: 6, Control: 28)
  Site 7: 89 subjects (ADHD: 0, Control: 89)


In [13]:
# Load FC subjects
fc_subjects = np.load(FC_DIR / "X_fc_subjects.npy", allow_pickle=True)

# Extract site info from FC subjects (assuming site is in subject ID or needs mapping)
print("\n" + "=" * 60)
print("FC SUBJECTS SAMPLE")
print("=" * 60)
print(f"Total FC subjects: {len(fc_subjects)}")
print(f"First 10 FC subjects: {fc_subjects[:10]}")


FC SUBJECTS SAMPLE
Total FC subjects: 955
First 10 FC subjects: ['0026001' '0026002' '0026004' '0026005' '0026009' '0026014' '0026015'
 '0026016' '0026017' '0026022']


In [14]:
# Find which labelled subjects are NOT in FC subjects
fc_set = set(fc_subjects)
labelled_set = set(subjects_all)

missing = labelled_set - fc_set
print("\n" + "=" * 60)
print("MISSING SUBJECTS (Labelled but no FC)")
print("=" * 60)
print(f"Total missing: {len(missing)}")

# Check sites of missing subjects
missing_sites = []
for subj in missing:
    idx = list(subjects_all).index(subj)
    missing_sites.append(sites_all[idx])

unique_missing_sites = np.unique(missing_sites)
print(f"Sites with missing subjects: {unique_missing_sites}")

for site in unique_missing_sites:
    count = sum(1 for s in missing_sites if s == site)
    print(f"  Site {site}: {count} missing")


MISSING SUBJECTS (Labelled but no FC)
Total missing: 300
Sites with missing subjects: ['4.0' '5' '6.0' '7']
  Site 4.0: 48 missing
  Site 5: 129 missing
  Site 6.0: 34 missing
  Site 7: 89 missing


In [4]:
import numpy as np
from pathlib import Path

ROOT = Path("data")
FC_DIR = ROOT / "03_fc_matrices"
RAW_DATA = ROOT / "RawDataBIDS"

# Load labels
y_all = np.load(FC_DIR / "y_binary.npy")
subjects_all = np.load(FC_DIR / "subjects.npy", allow_pickle=True)
sites_all = np.load(FC_DIR / "sites.npy", allow_pickle=True)

# Load FC subjects
fc_subjects = np.load(FC_DIR / "X_fc_subjects.npy", allow_pickle=True)

# Find missing subjects
fc_set = set(fc_subjects)
missing_subjects = []
missing_sites = []
missing_labels = []

for i, subj in enumerate(subjects_all):
    if subj not in fc_set:
        missing_subjects.append(subj)
        missing_sites.append(sites_all[i])
        missing_labels.append(y_all[i])

print("=" * 60)
print("MISSING SUBJECTS SUMMARY")
print("=" * 60)
print(f"Total missing: {len(missing_subjects)}")
print(f"Sites: {np.unique(missing_sites)}")

# Save missing subjects list
np.save(FC_DIR / "missing_subjects.npy", np.array(missing_subjects))
np.save(FC_DIR / "missing_sites.npy", np.array(missing_sites))
np.save(FC_DIR / "missing_labels.npy", np.array(missing_labels))

print(f"\n✅ Missing subjects list saved to {FC_DIR}")

MISSING SUBJECTS SUMMARY
Total missing: 300
Sites: ['4.0' '5' '6.0' '7']

✅ Missing subjects list saved to [DATA_ROOT]/03_fc_matrices


In [5]:
import nibabel as nib
from nilearn import image, input_data
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("CHECKING BOLD FILES FOR MISSING SUBJECTS")
print("=" * 60)

# Find all BOLD files
bold_files = list(RAW_DATA.glob("**/*_bold.nii.gz"))
bold_subjects = set()

for f in bold_files:
    for part in f.parts:
        if part.startswith('sub-'):
            subj_id = part.replace('sub-', '')
            bold_subjects.add(subj_id)
            break

print(f"Total BOLD subjects: {len(bold_subjects)}")

# Check which missing subjects have BOLD files
missing_with_bold = []
missing_no_bold = []

for subj in missing_subjects:
    if subj in bold_subjects:
        missing_with_bold.append(subj)
    else:
        missing_no_bold.append(subj)

print(f"\n✅ Missing subjects WITH BOLD files: {len(missing_with_bold)}")
print(f"❌ Missing subjects WITHOUT BOLD files: {len(missing_no_bold)}")

# Show which sites are missing BOLD files
missing_no_bold_sites = []
for subj in missing_no_bold:
    idx = list(subjects_all).index(subj)
    missing_no_bold_sites.append(sites_all[idx])

print(f"\nSites without BOLD files: {np.unique(missing_no_bold_sites)}")

/tmp/ipykernel_3289594/2796153038.py:2: FutureWarning: The import path 'nilearn.input_data' is deprecated in version 0.9. Importing from 'nilearn.input_data' will be possible at least until release 0.13.0. Please import from 'nilearn.maskers' instead.
  from nilearn import image, input_data


CHECKING BOLD FILES FOR MISSING SUBJECTS
Total BOLD subjects: 955

✅ Missing subjects WITH BOLD files: 0
❌ Missing subjects WITHOUT BOLD files: 300

Sites without BOLD files: ['4.0' '5' '6.0' '7']


In [6]:
import numpy as np
import nibabel as nib
from nilearn import image, input_data
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("data")
RAW_DATA = ROOT / "RawDataBIDS"
TS_DIR = ROOT / "02_timeseries"

# Load AAL atlas
atlas_path = ROOT / "RawDataBIDS" / "aal116MNI.nii.gz"
if not atlas_path.exists():
    print(f"❌ Atlas not found!")
    exit()

print("=" * 60)
print("EXTRACTING TIMESERIES FOR MISSING SUBJECTS")
print("=" * 60)

atlas_img = nib.load(atlas_path)
target_n_rois = 116

masker = input_data.NiftiLabelsMasker(
    atlas_img,
    standardize=True,
    detrend=True,
    low_pass=0.1,
    high_pass=0.01,
    t_r=2.0,
    memory='nilearn_cache',
    verbose=0
)

# Find BOLD files for missing subjects
missing_subject_set = set(missing_with_bold)
extracted = []
failed = []

for subj in tqdm(missing_with_bold, desc="Extracting"):
    try:
        # Find the BOLD file for this subject
        bold_file = None
        for f in RAW_DATA.glob(f"**/sub-{subj}/*_bold.nii.gz"):
            bold_file = f
            break
        
        if bold_file is None:
            failed.append((subj, "No BOLD file found"))
            continue
        
        # Extract timeseries
        img = image.load_img(str(bold_file))
        ts = masker.fit_transform(img)
        
        # Pad/truncate to 116 ROIs
        if ts.shape[1] < target_n_rois:
            padded_ts = np.zeros((ts.shape[0], target_n_rois))
            padded_ts[:, :ts.shape[1]] = ts
            ts = padded_ts
        elif ts.shape[1] > target_n_rois:
            ts = ts[:, :target_n_rois]
        
        # Save timeseries
        np.save(TS_DIR / f"{subj}_roi_timeseries.npy", ts)
        extracted.append(subj)
        
    except Exception as e:
        failed.append((subj, str(e)))

print(f"\n✅ Extracted: {len(extracted)} subjects")
print(f"❌ Failed: {len(failed)}")

if failed:
    print("\nFailed subjects:")
    for subj, err in failed[:10]:
        print(f"  {subj}: {err}")

EXTRACTING TIMESERIES FOR MISSING SUBJECTS


Extracting: 0it [00:00, ?it/s]


✅ Extracted: 0 subjects
❌ Failed: 0


In [7]:
import numpy as np

if FEATURES_DIR.exists():
    npy_files = list(FEATURES_DIR.glob("*.npy"))
    
    if npy_files:
        print("=" * 60)
        print("LOADING .NPY FILES FROM 04_features")
        print("=" * 60)
        
        for npy_file in npy_files[:5]:  # First 5 files
            try:
                data = np.load(npy_file, allow_pickle=True)
                print(f"\n📄 {npy_file.name}")
                print(f"   Shape: {data.shape}")
                print(f"   Type: {data.dtype}")
                
                # If it's a small array, show contents
                if data.size < 100:
                    print(f"   Data: {data}")
                
                # Check if it contains subject IDs
                if data.dtype == 'O' or data.dtype == '<U32' or data.dtype == 'U':
                    print(f"   Contains strings. First 5: {data[:5] if len(data) >= 5 else data}")
                    
            except Exception as e:
                print(f"\n❌ Error loading {npy_file.name}: {e}")
    else:
        print("No .npy files found in 04_features")

In [8]:
from pathlib import Path

ROOT = Path("data")

# Create new folders
folders = [
    "06_semi_supervised/data",
    "06_semi_supervised/models",
    "06_semi_supervised/results",
    "06_semi_supervised/notebooks",
    "07_pseudo_labeling/data",
    "07_pseudo_labeling/models",
    "07_pseudo_labeling/results",
    "07_pseudo_labeling/notebooks"
]

print("CREATING FOLDER STRUCTURE")

for folder in folders:
    path = ROOT / folder
    path.mkdir(parents=True, exist_ok=True)
    print(f"Created: {path}")

print("\nAll folders created successfully!")

CREATING FOLDER STRUCTURE
Created: [DATA_ROOT]/06_semi_supervised/data
Created: [DATA_ROOT]/06_semi_supervised/models
Created: [DATA_ROOT]/06_semi_supervised/results
Created: [DATA_ROOT]/06_semi_supervised/notebooks
Created: [DATA_ROOT]/07_pseudo_labeling/data
Created: [DATA_ROOT]/07_pseudo_labeling/models
Created: [DATA_ROOT]/07_pseudo_labeling/results
Created: [DATA_ROOT]/07_pseudo_labeling/notebooks

All folders created successfully!


In [9]:
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = Path("data")
FC_DIR = ROOT / "03_fc_matrices"
SS_DIR = ROOT / "06_semi_supervised/data"
PL_DIR = ROOT / "07_pseudo_labeling/data"

print("PREPARING DATA FOR BOTH APPROACHES")

# Load aligned data
X_fc = np.load(FC_DIR / "X_fc_aligned.npy")
y = np.load(FC_DIR / "y_aligned.npy")
subjects = np.load(FC_DIR / "aligned_subjects.npy", allow_pickle=True)
sites = np.load(FC_DIR / "aligned_sites.npy", allow_pickle=True)

print(f"Aligned data: {X_fc.shape}, {len(y)} subjects")

# Load unlabeled data
X_fc_full = np.load(FC_DIR / "X_fc.npy")
fc_subjects = np.load(FC_DIR / "X_fc_subjects.npy", allow_pickle=True)

# Get unlabeled subjects
labelled_set = set(subjects)
unlabeled_indices = [i for i, subj in enumerate(fc_subjects) if subj not in labelled_set]
X_unlabeled = X_fc_full[unlabeled_indices]
unlabeled_subjects = fc_subjects[unlabeled_indices]

print(f"Unlabeled data: {X_unlabeled.shape}, {len(unlabeled_subjects)} subjects")

# Flatten FC matrices for ML
n_rois = X_fc.shape[1]
X_flat = X_fc.reshape(X_fc.shape[0], -1)
X_unlabeled_flat = X_unlabeled.reshape(X_unlabeled.shape[0], -1)

print(f"\nFlattened shapes:")
print(f"  Labeled: {X_flat.shape}")
print(f"  Unlabeled: {X_unlabeled_flat.shape}")

# Split labeled data into train/val/test
X_train, X_temp, y_train, y_temp, subj_train, subj_temp, sites_train, sites_temp = train_test_split(
    X_flat, y, subjects, sites, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test, subj_val, subj_test, sites_val, sites_test = train_test_split(
    X_temp, y_temp, subj_temp, sites_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nData splits:")
print(f"  Train: {X_train.shape[0]} subjects")
print(f"  Val: {X_val.shape[0]} subjects")
print(f"  Test: {X_test.shape[0]} subjects")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
X_unlabeled_scaled = scaler.transform(X_unlabeled_flat)

print(f"\n Feature scaling complete")
# Save prepared data for both approaches

# For Semi-Supervised Learning
np.save(SS_DIR / "X_train.npy", X_train_scaled)
np.save(SS_DIR / "X_val.npy", X_val_scaled)
np.save(SS_DIR / "X_test.npy", X_test_scaled)
np.save(SS_DIR / "X_unlabeled.npy", X_unlabeled_scaled)
np.save(SS_DIR / "y_train.npy", y_train)
np.save(SS_DIR / "y_val.npy", y_val)
np.save(SS_DIR / "y_test.npy", y_test)
np.save(SS_DIR / "subjects_train.npy", subj_train)
np.save(SS_DIR / "subjects_val.npy", subj_val)
np.save(SS_DIR / "subjects_test.npy", subj_test)
np.save(SS_DIR / "sites_train.npy", sites_train)
np.save(SS_DIR / "sites_val.npy", sites_val)
np.save(SS_DIR / "sites_test.npy", sites_test)
np.save(SS_DIR / "unlabeled_subjects.npy", unlabeled_subjects)
np.save(SS_DIR / "scaler.npy", scaler)

# For Pseudo-Labeling (same data, separate folder)
np.save(PL_DIR / "X_train.npy", X_train_scaled)
np.save(PL_DIR / "X_val.npy", X_val_scaled)
np.save(PL_DIR / "X_test.npy", X_test_scaled)
np.save(PL_DIR / "X_unlabeled.npy", X_unlabeled_scaled)
np.save(PL_DIR / "y_train.npy", y_train)
np.save(PL_DIR / "y_val.npy", y_val)
np.save(PL_DIR / "y_test.npy", y_test)
np.save(PL_DIR / "subjects_train.npy", subj_train)
np.save(PL_DIR / "subjects_val.npy", subj_val)
np.save(PL_DIR / "subjects_test.npy", subj_test)
np.save(PL_DIR / "sites_train.npy", sites_train)
np.save(PL_DIR / "sites_val.npy", sites_val)
np.save(PL_DIR / "sites_test.npy", sites_test)
np.save(PL_DIR / "unlabeled_subjects.npy", unlabeled_subjects)
np.save(PL_DIR / "scaler.npy", scaler)

print(f"\nData saved to:")
print(f"   {SS_DIR}")
print(f"   {PL_DIR}")

# Print summary
print("DATA PREPARATION SUMMARY")

print(f"Labeled data:")
print(f"  Train: {len(y_train)} (ADHD: {sum(y_train)}, Control: {len(y_train)-sum(y_train)})")
print(f"  Val: {len(y_val)} (ADHD: {sum(y_val)}, Control: {len(y_val)-sum(y_val)})")
print(f"  Test: {len(y_test)} (ADHD: {sum(y_test)}, Control: {len(y_test)-sum(y_test)})")
print(f"\nUnlabeled data: {len(unlabeled_subjects)} subjects")
print(f"Total: {len(y_train) + len(y_val) + len(y_test) + len(unlabeled_subjects)} subjects")

# Site distribution in train set
print(f"\nSites in training set:")
for site in np.unique(sites_train):
    mask = sites_train == site
    print(f"  Site {site}: {sum(mask)} (ADHD: {sum(y_train[mask])}, Control: {sum(mask)-sum(y_train[mask])})")

PREPARING DATA FOR BOTH APPROACHES
Aligned data: (391, 116, 116), 391 subjects
Unlabeled data: (564, 116, 116), 564 subjects

Flattened shapes:
  Labeled: (391, 13456)
  Unlabeled: (564, 13456)

Data splits:
  Train: 273 subjects
  Val: 59 subjects
  Test: 59 subjects

 Feature scaling complete

Data saved to:
   [DATA_ROOT]/06_semi_supervised/data
   [DATA_ROOT]/07_pseudo_labeling/data
DATA PREPARATION SUMMARY
Labeled data:
  Train: 273 (ADHD: 112, Control: 161)
  Val: 59 (ADHD: 25, Control: 34)
  Test: 59 (ADHD: 24, Control: 35)

Unlabeled data: 564 subjects
Total: 955 subjects

Sites in training set:
  Site 1: 88 (ADHD: 28, Control: 60)
  Site 3: 60 (ADHD: 17, Control: 43)
  Site 5: 68 (ADHD: 41, Control: 27)
  Site 6: 57 (ADHD: 26, Control: 31)


In [10]:
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.semi_supervised import SelfTrainingClassifier
import joblib

ROOT = Path("data")
SS_DIR = ROOT / "06_semi_supervised"

print("SEMI-SUPERVISED LEARNING - SELF-TRAINING")

# Load data
X_train = np.load(SS_DIR / "data/X_train.npy")
y_train = np.load(SS_DIR / "data/y_train.npy")
X_val = np.load(SS_DIR / "data/X_val.npy")
y_val = np.load(SS_DIR / "data/y_val.npy")
X_test = np.load(SS_DIR / "data/X_test.npy")
y_test = np.load(SS_DIR / "data/y_test.npy")
X_unlabeled = np.load(SS_DIR / "data/X_unlabeled.npy")

print(f"Training: {X_train.shape[0]} subjects")
print(f"Validation: {X_val.shape[0]} subjects")
print(f"Test: {X_test.shape[0]} subjects")
print(f"Unlabeled: {X_unlabeled.shape[0]} subjects")

# Create combined dataset for self-training
X_combined = np.vstack([X_train, X_unlabeled])
y_combined = np.concatenate([y_train, np.full(X_unlabeled.shape[0], -1)])  # 1 = unlabeled

print(f"\nCombined dataset: {X_combined.shape[0]} subjects")
print(f"Labeled: {np.sum(y_combined != -1)}")
print(f"Unlabeled: {np.sum(y_combined == -1)}")

# Self-Training with Random Forest
base_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

self_training_model = SelfTrainingClassifier(
    base_model,
    threshold=0.75,
    criterion='threshold',
    k_best=10,
    max_iter=10,
    verbose=True
)

print("\n🔄 Training Self-Training model...")
self_training_model.fit(X_combined, y_combined)

# Evaluate
y_pred_val = self_training_model.predict(X_val)
y_pred_test = self_training_model.predict(X_test)

val_acc = accuracy_score(y_val, y_pred_val)
test_acc = accuracy_score(y_test, y_pred_test)

print("SELF-TRAINING RESULTS")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

print(f"\nTest Classification Report:")
print(classification_report(y_test, y_pred_test, target_names=['Control', 'ADHD']))

# Check how many unlabeled got labeled
n_labeled_final = np.sum(self_training_model.transduction_ != -1)
print(f"\n Final labeled subjects (including pseudo-labeled): {n_labeled_final}")
print(f"   Original labeled: {len(y_train)}")
print(f"   Newly labeled: {n_labeled_final - len(y_train)}")

# Save model
joblib.dump(self_training_model, SS_DIR / "models/self_training_model.pkl")
print(f"\nModel saved to: {SS_DIR}/models/self_training_model.pkl")

SEMI-SUPERVISED LEARNING - SELF-TRAINING
Training: 273 subjects
Validation: 59 subjects
Test: 59 subjects
Unlabeled: 564 subjects

Combined dataset: 837 subjects
Labeled: 273
Unlabeled: 564

🔄 Training Self-Training model...
End of iteration 1, added 4 new labels.
End of iteration 2, added 8 new labels.
End of iteration 3, added 12 new labels.
End of iteration 4, added 22 new labels.
End of iteration 5, added 25 new labels.
End of iteration 6, added 21 new labels.
End of iteration 7, added 25 new labels.
End of iteration 8, added 29 new labels.
End of iteration 9, added 34 new labels.
End of iteration 10, added 31 new labels.
SELF-TRAINING RESULTS
Validation Accuracy: 0.5763
Test Accuracy: 0.5932

Test Classification Report:
              precision    recall  f1-score   support

     Control       0.60      0.91      0.73        35
        ADHD       0.50      0.12      0.20        24

    accuracy                           0.59        59
   macro avg       0.55      0.52      0.46    

In [11]:
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

ROOT = Path("data")
PL_DIR = ROOT / "07_pseudo_labeling"

print("=" * 60)
print("PSEUDO-LABELING APPROACH")
print("=" * 60)

# Load data
X_train = np.load(PL_DIR / "data/X_train.npy")
y_train = np.load(PL_DIR / "data/y_train.npy")
X_val = np.load(PL_DIR / "data/X_val.npy")
y_val = np.load(PL_DIR / "data/y_val.npy")
X_test = np.load(PL_DIR / "data/X_test.npy")
y_test = np.load(PL_DIR / "data/y_test.npy")
X_unlabeled = np.load(PL_DIR / "data/X_unlabeled.npy")
unlabeled_subjects = np.load(PL_DIR / "data/unlabeled_subjects.npy", allow_pickle=True)

print(f"Training: {X_train.shape[0]} subjects")
print(f"Validation: {X_val.shape[0]} subjects")
print(f"Test: {X_test.shape[0]} subjects")
print(f"Unlabeled: {X_unlabeled.shape[0]} subjects")

# Step 1: Train initial model
print("\n📚 Step 1: Training initial model...")
initial_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
initial_model.fit(X_train, y_train)

# Evaluate initial model
y_pred_val = initial_model.predict(X_val)
y_pred_test = initial_model.predict(X_test)

print(f"Initial Test Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")

# Step 2: Generate pseudo-labels for unlabeled data
print("\n🔮 Step 2: Generating pseudo-labels...")
probabilities = initial_model.predict_proba(X_unlabeled)
predicted_labels = initial_model.predict(X_unlabeled)

# Get confidence scores
confidence = np.max(probabilities, axis=1)

# Set confidence threshold (e.g., 0.8)
threshold = 0.8
high_confidence_mask = confidence >= threshold

pseudo_labels = predicted_labels[high_confidence_mask]
pseudo_data = X_unlabeled[high_confidence_mask]
pseudo_subjects = unlabeled_subjects[high_confidence_mask]

print(f"   High-confidence predictions: {len(pseudo_labels)} / {len(X_unlabeled)}")
print(f"   Confidence threshold: {threshold}")
print(f"   ADHD predictions: {sum(pseudo_labels)}")
print(f"   Control predictions: {len(pseudo_labels) - sum(pseudo_labels)}")

# Step 3: Combine with original training data
print("\n📚 Step 3: Retraining with pseudo-labels...")
X_augmented = np.vstack([X_train, pseudo_data])
y_augmented = np.concatenate([y_train, pseudo_labels])

print(f"   Augmented training set: {X_augmented.shape[0]} subjects")
print(f"   Original: {len(y_train)}")
print(f"   Added: {len(pseudo_labels)}")

# Step 4: Train final model
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_augmented, y_augmented)

# Step 5: Evaluate final model
y_pred_val_final = final_model.predict(X_val)
y_pred_test_final = final_model.predict(X_test)

val_acc = accuracy_score(y_val, y_pred_val_final)
test_acc = accuracy_score(y_test, y_pred_test_final)

print("\n" + "=" * 60)
print("PSEUDO-LABELING RESULTS")
print("=" * 60)
print(f"Initial Test Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Final Test Accuracy: {test_acc:.4f}")
print(f"Improvement: {(test_acc - accuracy_score(y_test, y_pred_test)):.4f}")

print(f"\nFinal Test Classification Report:")
print(classification_report(y_test, y_pred_test_final, target_names=['Control', 'ADHD']))

# Step 6: Save predictions and model
# Save pseudo-labeled data
np.save(PL_DIR / "data/pseudo_labels.npy", pseudo_labels)
np.save(PL_DIR / "data/pseudo_subjects.npy", pseudo_subjects)
np.save(PL_DIR / "data/pseudo_confidence.npy", confidence[high_confidence_mask])

# Save model
joblib.dump(initial_model, PL_DIR / "models/initial_model.pkl")
joblib.dump(final_model, PL_DIR / "models/final_model.pkl")

print(f"\n✅ Models saved to: {PL_DIR}/models/")
print(f"   - initial_model.pkl")
print(f"   - final_model.pkl")

# Summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total labeled subjects used: {len(y_train)}")
print(f"Pseudo-labeled subjects added: {len(pseudo_labels)}")
print(f"Final training set: {len(y_augmented)}")
print(f"Validation set: {len(y_val)}")
print(f"Test set: {len(y_test)}")

PSEUDO-LABELING APPROACH
Training: 273 subjects
Validation: 59 subjects
Test: 59 subjects
Unlabeled: 564 subjects

📚 Step 1: Training initial model...
Initial Test Accuracy: 0.6271

🔮 Step 2: Generating pseudo-labels...
   High-confidence predictions: 2 / 564
   Confidence threshold: 0.8
   ADHD predictions: 0
   Control predictions: 2

📚 Step 3: Retraining with pseudo-labels...
   Augmented training set: 275 subjects
   Original: 273
   Added: 2

PSEUDO-LABELING RESULTS
Initial Test Accuracy: 0.6271
Final Test Accuracy: 0.5763
Improvement: -0.0508

Final Test Classification Report:
              precision    recall  f1-score   support

     Control       0.59      0.91      0.72        35
        ADHD       0.40      0.08      0.14        24

    accuracy                           0.58        59
   macro avg       0.50      0.50      0.43        59
weighted avg       0.51      0.58      0.48        59


✅ Models saved to: [DATA_ROOT]/07_pseudo_labeling/models/
   - initial_model.pkl
 

In [13]:
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("data")
PL_DIR = ROOT / "07_pseudo_labeling"

print("=" * 80)
print("FIXED PIPELINE: TESTING ALL APPROACHES")
print("=" * 80)

# Load data
X_train = np.load(PL_DIR / "data/X_train.npy")
y_train = np.load(PL_DIR / "data/y_train.npy")
X_val = np.load(PL_DIR / "data/X_val.npy")
y_val = np.load(PL_DIR / "data/y_val.npy")
X_test = np.load(PL_DIR / "data/X_test.npy")
y_test = np.load(PL_DIR / "data/y_test.npy")
X_unlabeled = np.load(PL_DIR / "data/X_unlabeled.npy")

print(f"Training: {X_train.shape[0]} subjects, {X_train.shape[1]} features")
print(f"Validation: {X_val.shape[0]} subjects")
print(f"Test: {X_test.shape[0]} subjects")
print(f"Unlabeled: {X_unlabeled.shape[0]} subjects")

# Check for NaN values
print(f"\nNaN check:")
print(f"  Training: {np.isnan(X_train).sum()} NaNs")
print(f"  Validation: {np.isnan(X_val).sum()} NaNs")
print(f"  Test: {np.isnan(X_test).sum()} NaNs")
print(f"  Unlabeled: {np.isnan(X_unlabeled).sum()} NaNs")

# Function to train and evaluate with NaN handling
def train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test, model, model_name):
    # Create pipeline with imputer + scaler + model
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])
    
    # Cross-validation
    try:
        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
    except:
        cv_mean = 0
        cv_std = 0
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred_val = pipeline.predict(X_val)
    y_pred_test = pipeline.predict(X_test)
    
    # Metrics
    val_acc = accuracy_score(y_val, y_pred_val)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test)
    
    try:
        test_auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
    except:
        test_auc = 0
    
    print(f"\n{model_name}:")
    print(f"  CV Accuracy: {cv_mean:.4f} (+/- {cv_std:.4f})")
    print(f"  Validation Accuracy: {val_acc:.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Test F1-Score: {test_f1:.4f}")
    print(f"  Test AUC: {test_auc:.4f}")
    
    return {
        'pipeline': pipeline,
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1': test_f1,
        'test_auc': test_auc,
        'cv_mean': cv_mean,
        'cv_std': cv_std,
        'y_pred_test': y_pred_test
    }

# Dictionary to store all results
all_results = {}

# APPROACH 1: BASELINE MODELS
print("\n" + "=" * 60)
print("APPROACH 1: BASELINE MODELS")
print("=" * 60)

baseline_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for name, model in baseline_models.items():
    result = train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test, model, name)
    all_results[f'baseline_{name}'] = result

# APPROACH 2: WITH CLASS WEIGHTS
print("\n" + "=" * 60)
print("APPROACH 2: CLASS WEIGHTS (Handle Imbalance)")
print("=" * 60)

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

weighted_models = {
    'LR (weighted)': LogisticRegression(max_iter=1000, class_weight=class_weight_dict, random_state=42),
    'RF (weighted)': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'GB (weighted)': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for name, model in weighted_models.items():
    result = train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test, model, name)
    all_results[f'weighted_{name}'] = result

# APPROACH 3: PCA DIMENSIONALITY REDUCTION
print("\n" + "=" * 60)
print("APPROACH 3: PCA DIMENSIONALITY REDUCTION")
print("=" * 60)

def evaluate_with_pca(n_comp):
    print(f"\n--- PCA with {n_comp} components ---")
    
    # Create pipeline with PCA
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n_comp, random_state=42)),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ])
    
    return train_and_evaluate_pipeline(pipeline, X_train, y_train, X_val, y_val, X_test, y_test, f'LR + PCA ({n_comp})')

def train_and_evaluate_pipeline(pipeline, X_train, y_train, X_val, y_val, X_test, y_test, model_name):
    # Cross-validation
    try:
        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
    except:
        cv_mean = 0
        cv_std = 0
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred_val = pipeline.predict(X_val)
    y_pred_test = pipeline.predict(X_test)
    
    # Metrics
    val_acc = accuracy_score(y_val, y_pred_val)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test)
    
    try:
        test_auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
    except:
        test_auc = 0
    
    print(f"  CV Accuracy: {cv_mean:.4f} (+/- {cv_std:.4f})")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Test F1-Score: {test_f1:.4f}")
    
    return {
        'pipeline': pipeline,
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1': test_f1,
        'test_auc': test_auc,
        'cv_mean': cv_mean,
        'cv_std': cv_std,
        'y_pred_test': y_pred_test
    }

for n_comp in [50, 100, 150, 200]:
    result = evaluate_with_pca(n_comp)
    all_results[f'pca_{n_comp}'] = result

# APPROACH 4: ENSEMBLE METHODS
print("\n" + "=" * 60)
print("APPROACH 4: ENSEMBLE METHODS")
print("=" * 60)

from sklearn.ensemble import VotingClassifier

# Create individual pipelines for ensemble
def create_pipeline(model):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])

ensemble_models = [
    ('lr', create_pipeline(LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))),
    ('rf', create_pipeline(RandomForestClassifier(n_estimators=50, max_depth=10, class_weight='balanced', random_state=42))),
    ('gb', create_pipeline(GradientBoostingClassifier(n_estimators=50, random_state=42)))
]

# Hard voting
voting_hard = VotingClassifier(estimators=ensemble_models, voting='hard')
result = train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test, voting_hard, 'Ensemble (Hard Voting)')
all_results['ensemble_hard'] = result

# Soft voting
voting_soft = VotingClassifier(estimators=ensemble_models, voting='soft')
result = train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test, voting_soft, 'Ensemble (Soft Voting)')
all_results['ensemble_soft'] = result

# APPROACH 5: SEMI-SUPERVISED (Self-Training)
print("\n" + "=" * 60)
print("APPROACH 5: SEMI-SUPERVISED (Self-Training)")
print("=" * 60)

from sklearn.semi_supervised import SelfTrainingClassifier

# Combine train and unlabeled
X_combined = np.vstack([X_train, X_unlabeled])
y_combined = np.concatenate([y_train, np.full(X_unlabeled.shape[0], -1)])

print(f"Combined: {X_combined.shape[0]} subjects (Labeled: {sum(y_combined != -1)}, Unlabeled: {sum(y_combined == -1)})")

# Base pipeline for self-training
base_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

self_train = SelfTrainingClassifier(
    base_pipeline,
    threshold=0.75,
    criterion='threshold',
    k_best=10,
    max_iter=10,
    verbose=True
)

# Train
self_train.fit(X_combined, y_combined)

# Evaluate
y_pred_val = self_train.predict(X_val)
y_pred_test = self_train.predict(X_test)

val_acc = accuracy_score(y_val, y_pred_val)
test_acc = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)

print(f"\nSelf-Training Results:")
print(f"  Validation Accuracy: {val_acc:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")
print(f"  Test F1-Score: {test_f1:.4f}")

all_results['self_training'] = {
    'pipeline': self_train,
    'val_acc': val_acc,
    'test_acc': test_acc,
    'test_f1': test_f1,
    'test_auc': 0,
    'cv_mean': 0,
    'cv_std': 0,
    'y_pred_test': y_pred_test
}

# COMPARE ALL RESULTS
print("\n" + "=" * 80)
print("FINAL COMPARISON: ALL APPROACHES")
print("=" * 80)

# Sort by test accuracy
sorted_results = sorted(all_results.items(), key=lambda x: x[1]['test_acc'], reverse=True)

print("\nRank | Approach | Test Acc | Val Acc | F1-Score | AUC | CV Mean")
print("-" * 80)

for i, (name, results) in enumerate(sorted_results, 1):
    print(f"{i:3d} | {name[:30]:30s} | {results['test_acc']:.4f}   | {results['val_acc']:.4f}   | {results['test_f1']:.4f}   | {results['test_auc']:.4f} | {results['cv_mean']:.4f}")

# SAVE BEST MODEL
best_name = sorted_results[0][0]
best_results = sorted_results[0][1]
best_model = best_results['pipeline']

print(f"\n✅ Best model: {best_name}")
print(f"   Test Accuracy: {best_results['test_acc']:.4f}")

# Save best model
joblib.dump(best_model, PL_DIR / "models/best_model.pkl")
print(f"✅ Best model saved to: {PL_DIR}/models/best_model.pkl")

# DETAILED REPORT FOR BEST MODEL
print("\n" + "=" * 60)
print("DETAILED REPORT: BEST MODEL")
print("=" * 60)

y_pred_best = best_results['y_pred_test']
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Control', 'ADHD']))

print("\n✅ Testing complete! All results saved.")

FIXED PIPELINE: TESTING ALL APPROACHES
Training: 273 subjects, 13456 features
Validation: 59 subjects
Test: 59 subjects
Unlabeled: 564 subjects

NaN check:
  Training: 422144 NaNs
  Validation: 70754 NaNs
  Test: 67374 NaNs
  Unlabeled: 1565368 NaNs

APPROACH 1: BASELINE MODELS

Logistic Regression:
  CV Accuracy: 0.5824 (+/- 0.0419)
  Validation Accuracy: 0.6949
  Test Accuracy: 0.6441
  Test F1-Score: 0.5333
  Test AUC: 0.6893

Random Forest:
  CV Accuracy: 0.6227 (+/- 0.0576)
  Validation Accuracy: 0.6102
  Test Accuracy: 0.6949
  Test F1-Score: 0.4000
  Test AUC: 0.7089

Gradient Boosting:
  CV Accuracy: 0.5754 (+/- 0.0395)
  Validation Accuracy: 0.5763
  Test Accuracy: 0.6102
  Test F1-Score: 0.4889
  Test AUC: 0.7333

APPROACH 2: CLASS WEIGHTS (Handle Imbalance)

LR (weighted):
  CV Accuracy: 0.5788 (+/- 0.0484)
  Validation Accuracy: 0.6949
  Test Accuracy: 0.6441
  Test F1-Score: 0.5333
  Test AUC: 0.6881

RF (weighted):
  CV Accuracy: 0.6335 (+/- 0.0443)
  Validation Accuracy:

In [14]:
import os
import subprocess
import torch

print("=" * 60)
print("PYTORCH CUDA CHECK")
print("=" * 60)

# PyTorch info
print("Torch version :", torch.__version__)
print("CUDA compiled :", torch.version.cuda)

# CUDA availability
print("CUDA available:", torch.cuda.is_available())

try:
    print("Device count  :", torch.cuda.device_count())

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i} Name   :", torch.cuda.get_device_name(i))

        # actual GPU computation test
        x = torch.randn(5000, 5000, device="cuda")
        y = torch.mm(x, x)

        print("GPU compute test: SUCCESS")
        print("Tensor device   :", y.device)

    else:
        print("GPU NOT AVAILABLE")

except Exception as e:
    print("CUDA ERROR:")
    print(e)

print("\n" + "=" * 60)
print("NVIDIA-SMI CHECK")
print("=" * 60)

try:
    result = subprocess.check_output(["nvidia-smi"]).decode()
    print(result)
except Exception as e:
    print("nvidia-smi failed:")
    print(e)

PYTORCH CUDA CHECK
Torch version : 2.5.1+cu121
CUDA compiled : 12.1
CUDA available: True
Device count  : 8
GPU 0 Name   : NVIDIA A100-SXM4-80GB
GPU 1 Name   : NVIDIA A100-SXM4-80GB
GPU 2 Name   : NVIDIA A100-SXM4-80GB
GPU 3 Name   : NVIDIA A100-SXM4-80GB
GPU 4 Name   : NVIDIA A100-SXM4-80GB
GPU 5 Name   : NVIDIA A100-SXM4-80GB
GPU 6 Name   : NVIDIA A100-SXM4-80GB
GPU 7 Name   : NVIDIA A100-SXM4-80GB
GPU compute test: SUCCESS
Tensor device   : cuda:0

NVIDIA-SMI CHECK
Sat Jun 27 17:32:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                   

In [7]:
# First, uninstall old versions
!pip uninstall torch-scatter torch-sparse -y

# Install for your PyTorch 2.12.0 with CUDA 13.0
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.12.0+cu130.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.12.0+cu130.html

# Reinstall torch-geometric (should be fine)
!pip install torch-geometric

In [7]:
# LOAD DATA (Add this at the start)
import numpy as np
from pathlib import Path

ROOT = Path("data")
FC_DIR = ROOT / "03_fc_matrices"

# Load labeled data
X_labeled = np.load(FC_DIR / "X_fc_aligned.npy")
y_labeled = np.load(FC_DIR / "y_aligned.npy")
subjects_labeled = np.load(FC_DIR / "aligned_subjects.npy", allow_pickle=True)
sites_labeled = np.load(FC_DIR / "aligned_sites.npy", allow_pickle=True)

print(f"Labeled: {X_labeled.shape[0]} subjects")
print(f"  ADHD: {sum(y_labeled)}, Control: {len(y_labeled) - sum(y_labeled)}")

# Load unlabeled data
X_full = np.load(FC_DIR / "X_fc.npy")
fc_subjects = np.load(FC_DIR / "X_fc_subjects.npy", allow_pickle=True)

labeled_set = set(subjects_labeled)
unlabeled_indices = [i for i, s in enumerate(fc_subjects) if s not in labeled_set]
X_unlabeled = X_full[unlabeled_indices]
subjects_unlabeled = fc_subjects[unlabeled_indices]

print(f"Unlabeled: {X_unlabeled.shape[0]} subjects")

Labeled: 391 subjects
  ADHD: 161, Control: 230
Unlabeled: 564 subjects


In [8]:
# IMPORTS
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
import joblib
import json

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import pennylane as qml

# GLOBAL SETUP
ROOT = Path("data")
FC_DIR = ROOT / "03_fc_matrices"
DATA_DIR = ROOT / "11_ensemble_labeling"
DATA_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
N_FEATURES = 2000
N_QUBITS = 8
N_LAYERS = 1
HIDDEN_DIM = 16
BATCH_SIZE = 2
EPOCHS = 50
WINDOW_SIZE = 30
STEP_SIZE = 10
DENSITY = 0.15

# PART 1: FEATURE EXTRACTION
def extract_upper_triangle_features(X):
    """Extract upper triangle of FC matrices (no diagonal, no duplicates)"""
    n_subjects = X.shape[0]
    n_rois = X.shape[1]
    n_features = n_rois * (n_rois - 1) // 2
    
    X_features = np.zeros((n_subjects, n_features))
    triu_idx = np.triu_indices_from(X[0], k=1)
    
    for i in range(n_subjects):
        X_features[i] = X[i][triu_idx]
    
    return X_features, triu_idx
from sklearn.impute import SimpleImputer

# After extracting features
X_labeled_feat, triu_idx = extract_upper_triangle_features(X_labeled)
X_unlabeled_feat, _ = extract_upper_triangle_features(X_unlabeled)

# Clean the data before normalization
initial_imputer = SimpleImputer(strategy='mean')
X_labeled_feat = initial_imputer.fit_transform(X_labeled_feat)
X_unlabeled_feat = initial_imputer.transform(X_unlabeled_feat)

print(f"NaNs after cleaning: {np.isnan(X_labeled_feat).sum()}")  # Should be 0
# PART 2: SITE INFERENCE
def infer_site(subject_id):
    """Infer imaging site from subject ID patterns"""
    subject_id = str(subject_id)
    if subject_id.startswith('002'):
        return 'NYU'
    elif subject_id.startswith('005'):
        return 'KKI'
    elif subject_id.startswith('008') or 'Peking' in subject_id:
        return 'Peking'
    elif 'OHSU' in subject_id or subject_id.startswith('004'):
        return 'OHSU'
    elif 'NeuroIMAGE' in subject_id or subject_id.startswith('006'):
        return 'NeuroIMAGE'
    elif 'Pittsburgh' in subject_id or subject_id.startswith('007'):
        return 'Pittsburgh'
    else:
        return 'Unknown'

# PART 3: LOAD DATA
print("\n" + "=" * 80)
print("LOADING DATA")
print("=" * 80)

# Load labeled data
X_labeled = np.load(FC_DIR / "X_fc_aligned.npy")
y_labeled = np.load(FC_DIR / "y_aligned.npy")
subjects_labeled = np.load(FC_DIR / "aligned_subjects.npy", allow_pickle=True)
sites_labeled = np.load(FC_DIR / "aligned_sites.npy", allow_pickle=True)

print(f"Labeled: {X_labeled.shape[0]} subjects")
print(f"  ADHD: {sum(y_labeled)}, Control: {len(y_labeled) - sum(y_labeled)}")

# Load unlabeled data
X_full = np.load(FC_DIR / "X_fc.npy")
fc_subjects = np.load(FC_DIR / "X_fc_subjects.npy", allow_pickle=True)

labeled_set = set(subjects_labeled)
unlabeled_indices = [i for i, s in enumerate(fc_subjects) if s not in labeled_set]
X_unlabeled = X_full[unlabeled_indices]
subjects_unlabeled = fc_subjects[unlabeled_indices]

print(f"Unlabeled: {X_unlabeled.shape[0]} subjects")

# Extract upper triangle features
X_labeled_feat, triu_idx = extract_upper_triangle_features(X_labeled)
X_unlabeled_feat, _ = extract_upper_triangle_features(X_unlabeled)

print(f"Features: {X_labeled_feat.shape[1]} (upper triangle)")

Using device: cuda
NaNs after cleaning: 0

LOADING DATA
Labeled: 391 subjects
  ADHD: 161, Control: 230
Unlabeled: 564 subjects
Features: 6670 (upper triangle)


In [10]:
# PART 4: SITE-AWARE NORMALIZATION WITH IMPUTATION
print("\n" + "=" * 80)
print("SITE-AWARE NORMALIZATION WITH IMPUTATION")
print("=" * 80)

from sklearn.impute import SimpleImputer

# First, clean the data globally
print("Cleaning NaNs from features...")
initial_imputer = SimpleImputer(strategy='mean')
X_labeled_feat = initial_imputer.fit_transform(X_labeled_feat)
X_unlabeled_feat = initial_imputer.transform(X_unlabeled_feat)
print(f"NaNs remaining: {np.isnan(X_labeled_feat).sum()} (should be 0)")

# ADD THIS CONSTANT FEATURES CHECK HERE
# Check for constant features (zero variance)
variances = np.var(X_labeled_feat, axis=0)
constant_features = np.where(variances == 0)[0]

if len(constant_features) > 0:
    print(f"Removing {len(constant_features)} constant features")
    # Remove constant features
    non_constant_mask = variances > 0
    X_labeled_feat = X_labeled_feat[:, non_constant_mask]
    X_unlabeled_feat = X_unlabeled_feat[:, non_constant_mask]
    print(f"New feature count: {X_labeled_feat.shape[1]}")
else:
    print("No constant features found")

# Now proceed with site-aware scaling
# Labeled: site-specific scaling
site_scalers = {}
X_labeled_normalized = np.zeros_like(X_labeled_feat)

for site in np.unique(sites_labeled):
    mask = sites_labeled == site
    if sum(mask) > 5:
        scaler = StandardScaler()
        X_labeled_normalized[mask] = scaler.fit_transform(X_labeled_feat[mask])
        site_scalers[site] = scaler
        print(f"  Site {site}: {sum(mask)} subjects")
# SITE INFERENCE FOR UNLABELED DATA
def infer_site(subject_id):
    """Infer imaging site from subject ID patterns"""
    subject_id = str(subject_id)
    if subject_id.startswith('002'):
        return 'NYU'
    elif subject_id.startswith('005'):
        return 'KKI'
    elif subject_id.startswith('008') or 'Peking' in subject_id:
        return 'Peking'
    elif 'OHSU' in subject_id or subject_id.startswith('004'):
        return 'OHSU'
    elif 'NeuroIMAGE' in subject_id or subject_id.startswith('006'):
        return 'NeuroIMAGE'
    elif 'Pittsburgh' in subject_id or subject_id.startswith('007'):
        return 'Pittsburgh'
    else:
        return 'Unknown'

# Get sites for unlabeled subjects
unlabeled_sites = np.array([infer_site(s) for s in subjects_unlabeled])
print(f"Unlabeled site distribution:")

# Initialize X_unlabeled_normalized BEFORE the loop
X_unlabeled_normalized = np.zeros_like(X_unlabeled_feat)

for site in np.unique(unlabeled_sites):
    mask = unlabeled_sites == site
    if site in site_scalers:
        X_unlabeled_normalized[mask] = site_scalers[site].transform(X_unlabeled_feat[mask])
        print(f"  Unlabeled site {site}: using labeled scaler")
    else:
        global_scaler = StandardScaler()
        global_scaler.fit(X_labeled_feat)
        X_unlabeled_normalized[mask] = global_scaler.transform(X_unlabeled_feat[mask])
        print(f"  Unlabeled site {site}: using global scaler")

print("✅ Normalization complete")

for site in np.unique(unlabeled_sites):
    print(f"  {site}: {sum(unlabeled_sites == site)}")
for site in np.unique(unlabeled_sites):
    mask = unlabeled_sites == site
    if site in site_scalers:
        X_unlabeled_normalized[mask] = site_scalers[site].transform(X_unlabeled_feat[mask])
        print(f"  Unlabeled site {site}: using labeled scaler")
    else:
        global_scaler = StandardScaler()
        global_scaler.fit(X_labeled_feat)
        X_unlabeled_normalized[mask] = global_scaler.transform(X_unlabeled_feat[mask])
        print(f"  Unlabeled site {site}: using global scaler")

print("✅ Normalization complete")

# PART 5: ENSEMBLE TRAINING
print("\n" + "=" * 80)
print("ENSEMBLE TRAINING")
print("=" * 80)
from sklearn.impute import SimpleImputer
def make_ensemble_pipeline(classifier, n_features=N_FEATURES):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),  # <-- Add this line
        ('feature_selection', SelectKBest(f_classif, k=n_features)),
        ('classifier', classifier)
    ])

models = {
    'rf': make_ensemble_pipeline(
        RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
    ),
    'gb': make_ensemble_pipeline(
        GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
    ),
    'lr': make_ensemble_pipeline(
        LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    ),
    'svm': make_ensemble_pipeline(
        CalibratedClassifierCV(
            SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
            method='sigmoid', cv=3
        )
    )
}

trained_models = {}
predictions = {}
cv_scores = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipeline in models.items():
    cv_scores[name] = cross_val_score(pipeline, X_labeled_normalized, y_labeled, cv=skf, scoring='accuracy')
    pipeline.fit(X_labeled_normalized, y_labeled)
    trained_models[name] = pipeline
    
    pred_proba = pipeline.predict_proba(X_unlabeled_normalized)
    pred = pipeline.predict(X_unlabeled_normalized)
    confidence = np.max(pred_proba, axis=1)
    
    predictions[name] = {'pred': pred, 'confidence': confidence}
    print(f"  {name}: CV = {cv_scores[name].mean():.4f}")

# PART 6: WEIGHTED ENSEMBLE
print("\n" + "=" * 80)
print("WEIGHTED ENSEMBLE")
print("=" * 80)

model_names = list(models.keys())
model_weights = np.array([cv_scores[name].mean() for name in model_names])
model_weights = model_weights / model_weights.sum()

all_preds = np.array([predictions[name]['pred'] for name in model_names]).T
all_confidences = np.array([predictions[name]['confidence'] for name in model_names]).T

weighted_votes = np.zeros(len(all_preds))
for i, weight in enumerate(model_weights):
    weighted_votes += weight * all_preds[:, i]

weighted_pred = (weighted_votes >= 0.5).astype(int)

weighted_consensus = np.zeros(len(weighted_pred))
for i in range(len(weighted_pred)):
    agreeing_weights = np.sum(model_weights[all_preds[i] == weighted_pred[i]])
    weighted_consensus[i] = agreeing_weights / np.sum(model_weights)

weighted_confidence = np.sum(all_confidences * model_weights, axis=1)

print(f"ADHD predicted: {sum(weighted_pred)}")
print(f"Control predicted: {len(weighted_pred) - sum(weighted_pred)}")

# PART 7: SELECTION MASK
print("\n" + "=" * 80)
print("SELECTION & VALIDATION")
print("=" * 80)

thresholds = {
    'high': {'consensus': 0.75, 'confidence': 0.7},
    'medium': {'consensus': 0.67, 'confidence': 0.6},
    'low': {'consensus': 0.50, 'confidence': 0.5}
}

best_level = 'medium'
selected_mask = (
    (weighted_consensus >= thresholds[best_level]['consensus']) & 
    (weighted_confidence >= thresholds[best_level]['confidence'])
)

selected_subjects = subjects_unlabeled[selected_mask]
selected_labels = weighted_pred[selected_mask]

print(f"Selected: {len(selected_subjects)} subjects")
print(f"  ADHD: {sum(selected_labels)}")
print(f"  Control: {len(selected_labels) - sum(selected_labels)}")

# PART 8: VALIDATION LOOP
if len(selected_subjects) > 0:
    # Split labeled data for validation
    X_temp, X_val_holdout, y_temp, y_val_holdout = train_test_split(
        X_labeled_normalized, y_labeled, test_size=0.2, random_state=42, stratify=y_labeled
    )
    
    # Baseline
    baseline_pipeline = Pipeline([
        ('scaler', 'passthrough'),
        ('feature_selection', SelectKBest(f_classif, k=N_FEATURES)),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    baseline_pipeline.fit(X_temp, y_temp)
    baseline_acc = accuracy_score(y_val_holdout, baseline_pipeline.predict(X_val_holdout))
    
    # With pseudo-labels
    X_augmented = np.vstack([X_temp, X_unlabeled_normalized[selected_mask]])
    y_augmented = np.concatenate([y_temp, selected_labels])
    
    aug_pipeline = Pipeline([
        ('scaler', 'passthrough'),
        ('feature_selection', SelectKBest(f_classif, k=N_FEATURES)),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    aug_pipeline.fit(X_augmented, y_augmented)
    aug_acc = accuracy_score(y_val_holdout, aug_pipeline.predict(X_val_holdout))
    
    print(f"Baseline: {baseline_acc:.4f}")
    print(f"Augmented: {aug_acc:.4f}")
    
    use_pseudo = aug_acc >= baseline_acc
    print(f"Use pseudo-labels: {use_pseudo}")
else:
    use_pseudo = False
    print("No pseudo-labels selected")


SITE-AWARE NORMALIZATION WITH IMPUTATION
Cleaning NaNs from features...
NaNs remaining: 0 (should be 0)
No constant features found
  Site 1: 136 subjects
  Site 3: 83 subjects
  Site 5: 93 subjects
  Site 6: 79 subjects
Unlabeled site distribution:
  Unlabeled site NYU: using global scaler
  Unlabeled site Unknown: using global scaler
✅ Normalization complete
  NYU: 135
  Unknown: 429
  Unlabeled site NYU: using global scaler
  Unlabeled site Unknown: using global scaler
✅ Normalization complete

ENSEMBLE TRAINING
  rf: CV = 0.6419
  gb: CV = 0.6521
  lr: CV = 0.5498
  svm: CV = 0.6420

WEIGHTED ENSEMBLE
ADHD predicted: 110
Control predicted: 454

SELECTION & VALIDATION
Selected: 484 subjects
  ADHD: 86
  Control: 398
Baseline: 0.6709
Augmented: 0.7215
Use pseudo-labels: True


In [ ]:
# PART 9: PRODUCTION EXPORT (SAVE BOTH FORMATS)
print("\n" + "=" * 80)
print("PRODUCTION EXPORT")
print("=" * 80)

# Fit production selector for QGCNN features
production_selector = SelectKBest(f_classif, k=N_FEATURES)
production_selector.fit(X_labeled_normalized, y_labeled)

if use_pseudo and len(selected_subjects) > 0:
    # Get full upper triangle features (6670) for graph reconstruction
    X_labeled_full = X_labeled_normalized  # Already normalized, full 6670
    X_unlabeled_full = X_unlabeled_normalized[selected_mask]  # Full 6670
    
    # Get reduced features (2000) for QGCNN input
    X_labeled_reduced = production_selector.transform(X_labeled_normalized)
    X_unlabeled_reduced = production_selector.transform(X_unlabeled_normalized[selected_mask])
    
    # Combine both versions
    X_combined_full = np.vstack([X_labeled_full, X_unlabeled_full])
    X_combined_reduced = np.vstack([X_labeled_reduced, X_unlabeled_reduced])
    y_combined = np.concatenate([y_labeled, selected_labels])
    subjects_combined = np.concatenate([subjects_labeled, selected_subjects])
    
    print(f"Combined full (6670): {X_combined_full.shape}")
    print(f"Combined reduced (2000): {X_combined_reduced.shape}")
else:
    X_combined_full = X_labeled_normalized
    X_combined_reduced = production_selector.transform(X_labeled_normalized)
    y_combined = y_labeled
    subjects_combined = subjects_labeled

# Save BOTH versions
np.save(DATA_DIR / "X_combined_full.npy", X_combined_full)      # For graph reconstruction
np.save(DATA_DIR / "X_combined_reduced.npy", X_combined_reduced) # For QGCNN node features
np.save(DATA_DIR / "y_combined.npy", y_combined)
np.save(DATA_DIR / "subjects_combined.npy", subjects_combined)
joblib.dump(production_selector, DATA_DIR / "production_selector.pkl")

print(f"Saved {len(y_combined)} subjects to {DATA_DIR}")
print(f"  ADHD: {sum(y_combined)}")
print(f"  Control: {len(y_combined) - sum(y_combined)}")
# PART 10: QGCNN GRAPH CONSTRUCTION
def prepare_qgcnn_graphs(X_features, y_labels, n_rois=116, density=0.15):
    """Convert FC features to PyG graphs"""
    n_samples = X_features.shape[0]
    graphs = []
    triu_idx_local = np.triu_indices(n_rois, k=1)
    
    for i in tqdm(range(n_samples), desc="Building graphs", disable=n_samples>500):
        fc_matrix = np.zeros((n_rois, n_rois))
        fc_matrix[triu_idx_local] = X_features[i]
        fc_matrix = fc_matrix + fc_matrix.T
        
        abs_fc = np.abs(fc_matrix)
        upper_vals = abs_fc[triu_idx_local]
        thresh_value = np.percentile(upper_vals, 100 * (1 - density))
        
        edge_mask = abs_fc >= thresh_value
        edge_index = torch.tensor(np.array(np.where(edge_mask)), dtype=torch.long)
        edge_attr = torch.tensor(fc_matrix[edge_mask], dtype=torch.float32)
        
        degree = np.sum(abs_fc >= thresh_value, axis=1, keepdims=True) / n_rois
        node_features = np.hstack([fc_matrix, degree])
        node_features = torch.tensor(node_features, dtype=torch.float32)
        
        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor(y_labels[i], dtype=torch.long)
        )
        graphs.append(data)
    
    return graphs

# PART 11: DATA SPLITTING (USE FULL FEATURES)
print("\n" + "=" * 80)
print("DATA SPLITTING (CLEAN TEST SET)")
print("=" * 80)

# Load the FULL features (6670) for graph reconstruction
X_combined_full = np.load(DATA_DIR / "X_combined_full.npy")
y_combined = np.load(DATA_DIR / "y_combined.npy")
subjects_combined = np.load(DATA_DIR / "subjects_combined.npy", allow_pickle=True)

# Load REDUCED features (2000) for QGCNN node features later
X_combined_reduced = np.load(DATA_DIR / "X_combined_reduced.npy")

# Separate clean labeled vs pseudo-labeled
n_clean = len(y_labeled)
X_clean_full = X_combined_full[:n_clean]
y_clean = y_combined[:n_clean]
subjects_clean = subjects_combined[:n_clean]

if len(X_combined_full) > n_clean:
    X_pseudo_full = X_combined_full[n_clean:]
    y_pseudo = y_combined[n_clean:]
    subjects_pseudo = subjects_combined[n_clean:]
else:
    X_pseudo_full = np.array([])
    y_pseudo = np.array([])
    subjects_pseudo = np.array([])

print(f"Clean labeled: {len(y_clean)}")
print(f"Pseudo-labeled: {len(y_pseudo)}")

# Split CLEAN labeled data for test set (using FULL features)
X_train_clean_full, X_test_clean_full, y_train_clean, y_test_clean, subj_train, subj_test = train_test_split(
    X_clean_full, y_clean, subjects_clean,
    test_size=0.2, random_state=42, stratify=y_clean
)

# Split further for validation
X_train_clean_full, X_val_clean_full, y_train_clean, y_val_clean, subj_train, subj_val = train_test_split(
    X_train_clean_full, y_train_clean, subj_train,
    test_size=0.2, random_state=42, stratify=y_train_clean
)

print(f"Clean splits (using FULL 6670 features):")
print(f"  Train: {len(y_train_clean)}")
print(f"  Val: {len(y_val_clean)}")
print(f"  Test: {len(y_test_clean)}")

# Combine clean training with pseudo-labels (FULL features)
if len(y_pseudo) > 0 and use_pseudo:
    X_train_full = np.vstack([X_train_clean_full, X_pseudo_full])
    y_train = np.concatenate([y_train_clean, y_pseudo])
    print(f"Training data: {len(y_train_clean)} clean + {len(y_pseudo)} pseudo = {len(y_train)}")
else:
    X_train_full = X_train_clean_full
    y_train = y_train_clean
    print(f"Training data: {len(y_train)} clean only")

# Save reduced versions for QGCNN input later
# (Match the same indices from full version)
train_indices = list(range(len(X_train_full)))
val_indices = list(range(len(X_val_clean_full)))
test_indices = list(range(len(X_test_clean_full)))

# For now, use FULL features for graph building
# PART 12: BUILD GRAPHS (USING FULL FEATURES)
print("\n" + "=" * 80)
print("BUILDING GRAPHS")
print("=" * 80)

# These now use the FULL 6670 features
train_graphs = prepare_qgcnn_graphs(X_train_full, y_train, density=DENSITY)
val_graphs = prepare_qgcnn_graphs(X_val_clean_full, y_val_clean, density=DENSITY)
test_graphs = prepare_qgcnn_graphs(X_test_clean_full, y_test_clean, density=DENSITY)

print(f"Train graphs: {len(train_graphs)}")
print(f"Val graphs: {len(val_graphs)}")
print(f"Test graphs: {len(test_graphs)} (CLEAN ONLY!)")

In [21]:
# PART 13: QUANTUM CIRCUIT SETUP (FIXED)
print("\n" + "=" * 80)
print("QUANTUM CIRCUIT SETUP")
print("=" * 80)

# Use CPU version (stable, no GPU OOM)
dev = qml.device("lightning.qubit", wires=N_QUBITS)
print(f"Device: {dev.name} (CPU)")
print(f"Qubits: {N_QUBITS}")
print(f"Diff method: adjoint")

@qml.qnode(dev, interface="torch", diff_method="adjoint")
def quantum_embedding_circuit(inputs, weights):
    """Quantum embedding circuit with angle encoding and variational layers"""
    # Angle encoding
    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[(i + N_QUBITS) % N_QUBITS], wires=i)
    
    # Variational layers
    for layer in range(N_LAYERS):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
            qml.RX(weights[layer, i, 2], wires=i)
        
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[N_QUBITS - 1, 0])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

# Test the circuit
test_inputs = torch.randn(N_QUBITS * 2)
test_weights = torch.randn(N_LAYERS, N_QUBITS, 3)
try:
    test_output = quantum_embedding_circuit(test_inputs, test_weights)
    print(f"✅ Circuit test successful! Output features: {len(test_output)}")
except Exception as e:
    print(f"❌ Circuit test failed: {e}")
# PART 14: QUANTUM EMBEDDING LAYER
class QuantumEmbedding(nn.Module):
    def __init__(self, input_dim=117, n_qubits=N_QUBITS, n_layers=N_LAYERS):
        super().__init__()
        self.n_qubits = n_qubits
        self.classical_proj = nn.Linear(input_dim, n_qubits * 2)
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.quantum_layer = qml.qnn.TorchLayer(quantum_embedding_circuit, weight_shapes)
        
    def forward(self, x):
        projected = self.classical_proj(x)
        scaled_inputs = torch.tanh(projected) * np.pi
        return self.quantum_layer(scaled_inputs)

# PART 15: HYBRID QGCNN
class HybridQGCNN(nn.Module):
    def __init__(self, input_dim=117, n_qubits=N_QUBITS, n_layers=N_LAYERS, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.quantum_embed = QuantumEmbedding(input_dim, n_qubits, n_layers)
        self.gcn1 = GCNConv(n_qubits, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)
        self.gcn3 = GCNConv(hidden_dim, 16)
        self.fc = nn.Linear(16, 2)
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        
    def forward(self, data):
        # Move tensors individually (not the whole batch container)
        x = data.x.to(device)
        edge_index = data.edge_index.to(device)
        edge_attr = data.edge_attr.to(device)
        batch = data.batch.to(device)
        
        quantum_emb = self.quantum_embed(x)
        
        x = self.gcn1(quantum_emb, edge_index, edge_attr)
        x = F.relu(x)
        x = self.bn1(x)
        x = self.dropout(x)
        
        x = self.gcn2(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.bn2(x)
        x = self.dropout(x)
        
        x = self.gcn3(x, edge_index, edge_attr)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)
# PART 16: TRAINING LOOP
def train_qgcnn(model, train_loader, val_loader, epochs=EPOCHS, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    best_val_acc = 0
    best_model_state = None
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            # REMOVED: batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = F.cross_entropy(out, batch.y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, pred = out.max(dim=1)
            train_correct += (pred == batch.y).sum().item()
            train_total += batch.y.size(0)
        
        train_acc = train_correct / train_total
        
        model.eval()
        val_preds = []
        val_labels = []
        val_probs = []
        
        with torch.no_grad():
            for batch in val_loader:
                # REMOVED: batch.to(device)
                out = model(batch)
                probs = F.softmax(out, dim=1)
                
                _, pred = out.max(dim=1)
                val_preds.extend(pred.cpu().numpy())
                val_labels.extend(batch.y.cpu().numpy())
                val_probs.extend(probs[:, 1].cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds)
        val_auc = roc_auc_score(val_labels, val_probs)
        
        scheduler.step(val_acc)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, F1={val_f1:.4f}, AUC={val_auc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            torch.save(best_model_state, DATA_DIR / "best_qgcnn.pth")
    
    return model, best_val_acc
print("moving to `17")


QUANTUM CIRCUIT SETUP
Device: lightning.qubit (CPU)
Qubits: 12
Diff method: adjoint
✅ Circuit test successful! Output features: 12
moving to `17


In [22]:
# Check GPU memory before training
print(f"GPU 0 memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU 0 memory cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

# Check batch size and graph sizes
print(f"Batch size: {BATCH_SIZE}")
print(f"Train graphs: {len(train_graphs)}")
print(f"Sample graph nodes: {train_graphs[0].x.shape[0]}")
print(f"Sample graph edges: {train_graphs[0].edge_index.shape[1]}")

# Test loading a single batch
print("Testing single batch load...")
test_loader = DataLoader(train_graphs, batch_size=1, shuffle=False)
try:
    for batch in test_loader:
        batch.to(device)
        print(f"✅ Single batch loaded! Batch size: {batch.x.shape}")
        break
except Exception as e:
    print(f"❌ Failed to load batch: {e}")

# PART 17: RUN TRAINING
print("\n" + "=" * 80)
print("RUNNING QGCNN TRAINING")
print("=" * 80)

train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

model = HybridQGCNN()
model, best_val_acc = train_qgcnn(model, train_loader, val_loader, epochs=EPOCHS)

print(f"\n✅ Best Validation Accuracy: {best_val_acc:.4f}")

# PART 18: TEST EVALUATION (CLEAN ONLY)
print("\n" + "=" * 80)
print("FINAL TEST EVALUATION (CLEAN LABELS ONLY)")
print("=" * 80)

# Load best model
model.load_state_dict(torch.load(DATA_DIR / "best_qgcnn.pth"))
model.eval()

test_preds = []
test_labels = []
test_probs = []

with torch.no_grad():
    for batch in test_loader:
        batch.to(device)
        out = model(batch)
        probs = F.softmax(out, dim=1)
        
        _, pred = out.max(dim=1)
        test_preds.extend(pred.cpu().numpy())
        test_labels.extend(batch.y.cpu().numpy())
        test_probs.extend(probs[:, 1].cpu().numpy())

test_acc = accuracy_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds)
test_auc = roc_auc_score(test_labels, test_probs)

print(f"\n✅ FINAL RESULTS (Clean Test Set):")
print(f"  Test Accuracy: {test_acc:.4f}")
print(f"  Test F1-Score: {test_f1:.4f}")
print(f"  Test AUC: {test_auc:.4f}")

print(f"\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=['Control', 'ADHD']))

# PART 19: SAVE FINAL RESULTS
results = {
    'test_accuracy': float(test_acc),
    'test_f1': float(test_f1),
    'test_auc': float(test_auc),
    'best_val_accuracy': float(best_val_acc),
    'n_qubits': N_QUBITS,
    'n_layers': N_LAYERS,
    'hidden_dim': HIDDEN_DIM,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'density': DENSITY,
    'pseudo_labels_used': int(len(y_pseudo)) if use_pseudo else 0,
    'total_training_samples': len(y_train),
    'clean_test_samples': len(y_test_clean)
}

with open(DATA_DIR / "qgcnn_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to: {DATA_DIR}/qgcnn_results.json")
print("\n🎉 PIPELINE COMPLETE!")

GPU 0 memory allocated: 0.00 GB
GPU 0 memory cached: 0.00 GB
Batch size: 2
Train graphs: 733
Sample graph nodes: 116
Sample graph edges: 2002
Testing single batch load...
❌ Failed to load batch: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


RUNNING QGCNN TRAINING


Epoch 1:   0%|          | 0/367 [00:00<?, ?it/s]


In [6]:
import torch
import sys
import subprocess

print("=" * 60)
print("SYSTEM & HARDWARE CHECK")
print("=" * 60)

# 1. PyTorch version
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}:")
        print(f"  Name: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
        print(f"  Memory allocated: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
        print(f"  Memory cached: {torch.cuda.memory_reserved(i) / 1e9:.2f} GB")

# 2. Python environment
print(f"\nPython executable: {sys.executable}")
print(f"Python version: {sys.version}")

# 3. Check if using GPU 0 only or all GPUs
print(f"\nDefault device: {torch.cuda.current_device()}")

# 4. Check if DataParallel is being used
import torch.nn as nn
if hasattr(torch, 'nn') and hasattr(nn, 'DataParallel'):
    print("DataParallel available: Yes")

SYSTEM & HARDWARE CHECK

PyTorch version: 2.7.1+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1

GPU 0:
  Name: NVIDIA A100-SXM4-80GB
  Memory: 85.1 GB
  Memory allocated: 0.00 GB
  Memory cached: 0.00 GB

Python executable: /home/nvidia/.venv/bin/python3
Python version: 3.12.13 (main, Mar 24 2026, 22:49:22) [Clang 22.1.1 ]

Default device: 0
DataParallel available: Yes


In [7]:
import os
import torch
import gc

# Force both PyTorch and PennyLane to use ONLY GPU 0
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Clear any lingering allocations
torch.cuda.empty_cache()
gc.collect()

# Set device globally
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"✅ Hardware Aligned. Target: {device}")

✅ Hardware Aligned. Target: cuda:0


In [14]:
# QUANTUM CIRCUIT SETUP (CPU FALLBACK)
import pennylane as qml
import numpy as np

N_QUBITS = 12
N_LAYERS = 3

# Use CPU version to avoid GPU OOM
dev = qml.device("lightning.qubit", wires=N_QUBITS)
print(f"Device: {dev.name} (CPU)")
print(f"Qubits: {N_QUBITS}")
print(f"Diff method: adjoint")

@qml.qnode(dev, interface="torch", diff_method="adjoint")
def quantum_embedding_circuit(inputs, weights):
    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[(i + N_QUBITS) % N_QUBITS], wires=i)
    
    for layer in range(N_LAYERS):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
            qml.RX(weights[layer, i, 2], wires=i)
        
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[N_QUBITS - 1, 0])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

# Test the circuit
test_inputs = torch.randn(N_QUBITS * 2)
test_weights = torch.randn(N_LAYERS, N_QUBITS, 3)
try:
    test_output = quantum_embedding_circuit(test_inputs, test_weights)
    print(f"✅ Circuit test successful! Output features: {len(test_output)}")
except Exception as e:
    print(f"❌ Circuit test failed: {e}")

Device: lightning.qubit (CPU)
Qubits: 12
Diff method: adjoint
✅ Circuit test successful! Output features: 12


In [9]:
import pennylane as qml
import torch

print(qml.__version__)
print(torch.__version__)

0.45.1
2.7.1+cu126


## Execution Summary & Provenance

### Audited Scientific Findings
- **Key Results**: Procedure I (notebook executed outputs) generated 552 high-confidence pseudo-labels; Procedure II ensemble generated 484 pseudo-labels. Distinct from Exp 07's 713 pseudo-labels.
- **Primary Result Artifact**: `results/exp06/pseudo_labels_summary.csv`

### Known Limitations & Methodological Constraints
- Pseudo-label quality depends on conservative confidence thresholding (margin >= 0.20).
- For full reproduction guidelines and dataset acquisition steps, see [reproduction.md](../../docs/reproduction.md).

